# TAR 복원

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from tqdm.auto import tqdm
import tarfile

BASE = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "AI개론/261110스터디/multiclass_yolo26n"
)

TAR_PATH = BASE / "dataset_raw_3991_v1.tar"
MODEL_DIR = BASE / "models"

WORK_ROOT = Path("/content/multiclass_yolo26n")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_TAR = Path("/content/dataset_raw_3991_v1.tar")

assert TAR_PATH.is_file(), f"TAR 없음: {TAR_PATH}"
assert MODEL_DIR.is_dir(), f"모델 폴더 없음: {MODEL_DIR}"

# 1. Drive -> Colab 로컬 복사
if (not LOCAL_TAR.exists()
        or LOCAL_TAR.stat().st_size != TAR_PATH.stat().st_size):

    with TAR_PATH.open("rb") as src, LOCAL_TAR.open("wb") as dst:
        with tqdm(
            total=TAR_PATH.stat().st_size,
            desc="TAR 로컬 복사",
            unit="B",
            unit_scale=True
        ) as bar:

            while True:
                chunk = src.read(8 * 1024 * 1024)
                if not chunk:
                    break

                dst.write(chunk)
                bar.update(len(chunk))

# 2. 이미 압축을 풀었다면 재사용
def find_dataset_root():
    candidates = [
        WORK_ROOT,
        WORK_ROOT / "dataset_raw"
    ]

    candidates += [
        p.parent.parent
        for p in WORK_ROOT.rglob("images/train")
    ]

    for root in candidates:
        if (
            (root / "images/train").is_dir()
            and (root / "labels/train").is_dir()
        ):
            return root

    return None

RAW_ROOT = find_dataset_root()

if RAW_ROOT is None:
    with tarfile.open(LOCAL_TAR, "r:*") as tf:
        members = tf.getmembers()

        for member in tqdm(
            members,
            desc="TAR 압축 해제",
            unit="file"
        ):
            tf.extract(
                member,
                path=WORK_ROOT,
                filter="data"
            )

    RAW_ROOT = find_dataset_root()

assert RAW_ROOT is not None, "압축 해제 후 데이터셋을 찾지 못했습니다."

for split in ("train", "val", "test"):
    assert (RAW_ROOT / "images" / split).is_dir()
    assert (RAW_ROOT / "labels" / split).is_dir()

print("데이터셋 준비 완료:", RAW_ROOT)
print("모델 폴더:", MODEL_DIR)


Mounted at /content/drive


TAR 로컬 복사:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

TAR 압축 해제:   0%|          | 0/7992 [00:00<?, ?file/s]

데이터셋 준비 완료: /content/multiclass_yolo26n/dataset_raw
모델 폴더: /content/drive/MyDrive/Colab Notebooks/AI개론/261110스터디/multiclass_yolo26n/models


# 기존 PyTorch 체크포인트 로드

In [ ]:

%pip -q install ultralytics

import torch
import torch.nn as nn

from torchvision.models import resnet18
from ultralytics import YOLO

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

HEAD_DIMS = {
    "class_id": 5,
    "damage": 2,
    "dirtiness": 2,
    "cover": 2,
    "transparency": 2
}

class MultiTaskResNet18(nn.Module):
    def __init__(self, dropout_p=0.2):
        super().__init__()

        self.backbone = resnet18(weights=None)
        feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.heads = nn.ModuleDict({
            name: nn.Sequential(
                nn.Dropout(dropout_p),
                nn.Linear(feature_dim, count)
            )
            for name, count in HEAD_DIMS.items()
        })

    def forward(self, x):
        features = self.backbone(x)
        return {
            name: head(features)
            for name, head in self.heads.items()
        }

YOLO_PATH = MODEL_DIR / "yolo26n_raw_retrain_best.pt"

RESNET_PATH = (
    MODEL_DIR /
    "resnet18_multitask_raw_roi7579_v1_best.pt"
)

assert YOLO_PATH.is_file(), YOLO_PATH
assert RESNET_PATH.is_file(), RESNET_PATH

# 기존 YOLO26n 로드
yolo = YOLO(str(YOLO_PATH))

# 직접 생성한 신뢰할 수 있는 PyTorch 파일만 로드
checkpoint = torch.load(
    RESNET_PATH,
    map_location="cpu",
    weights_only=False
)

resnet = MultiTaskResNet18(dropout_p=0.2)
resnet.load_state_dict(checkpoint["model_state"])

resnet = resnet.to(DEVICE).eval()

print("YOLO26n 로드 완료")
print("ResNet18 로드 완료")
print("ResNet18 best epoch:", checkpoint.get("epoch"))
print("Device:", DEVICE)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
YOLO26n 로드 완료
ResNet18 로드 완료
ResNet18 best epoch: 2
Device: cuda


# GPU와 기존 체크포인트 확인

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2
    ),
    "GB"
)

print("RAW dataset:", RAW_ROOT)
print("YOLO model:", YOLO_PATH)
print("ResNet model:", RESNET_PATH)
print("ResNet heads:", list(resnet.heads.keys()))

GPU: NVIDIA L4
VRAM: 22.03 GB
RAW dataset: /content/multiclass_yolo26n/dataset_raw
YOLO model: /content/drive/MyDrive/Colab Notebooks/AI개론/261110스터디/multiclass_yolo26n/models/yolo26n_raw_retrain_best.pt
ResNet model: /content/drive/MyDrive/Colab Notebooks/AI개론/261110스터디/multiclass_yolo26n/models/resnet18_multitask_raw_roi7579_v1_best.pt
ResNet heads: ['class_id', 'damage', 'dirtiness', 'cover', 'transparency']


# Grad-CAM 구현

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random

from pathlib import Path
from tqdm.auto import tqdm
from torchvision import transforms

CLASS_NAMES = {
    0: "styrofoam",
    1: "vinyl",
    2: "brown_glass",
    3: "green_glass",
    4: "clear_glass"
}

CAM_DIR = BASE / "experiments_v3" / "cam_analysis"
CAM_DIR.mkdir(parents=True, exist_ok=True)

normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

def prepare_roi(image_bgr, size=224):
    """기존 ResNet18 학습 시 사용한 Letterbox 방식."""

    h, w = image_bgr.shape[:2]
    scale = min(size / w, size / h)

    nw = max(1, round(w * scale))
    nh = max(1, round(h * scale))

    resized = cv2.resize(
        image_bgr,
        (nw, nh),
        interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    )

    canvas = np.full(
        (size, size, 3),
        114,
        dtype=np.uint8
    )

    x = (size - nw) // 2
    y = (size - nh) // 2

    canvas[y:y+nh, x:x+nw] = resized

    # 기존 ROI 학습 파일의 JPEG 저장 특성 반영
    ok, buffer = cv2.imencode(
        ".jpg", canvas,
        [cv2.IMWRITE_JPEG_QUALITY, 95]
    )
    assert ok
    canvas = cv2.imdecode(buffer, cv2.IMREAD_COLOR)

    rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)

    tensor = normalize(
        transforms.ToTensor()(rgb)
    ).unsqueeze(0).to(DEVICE)

    return canvas, tensor


class GradCAM:
    def __init__(self, model):
        self.model = model
        self.activations = None
        self.gradients = None

        self.handle = model.backbone.layer4[-1].register_forward_hook(
            self._forward_hook
        )

    def _forward_hook(self, module, inputs, output):
        self.activations = output.detach()
        output.register_hook(self._save_gradient)

    def _save_gradient(self, gradient):
        self.gradients = gradient.detach()

    def generate(self, x, target_class=None):

        self.model.eval()
        self.model.zero_grad(set_to_none=True)

        with torch.enable_grad():
            outputs = self.model(x)
            logits = outputs["class_id"]

            probabilities = torch.softmax(logits, dim=1)[0]

            predicted_class = int(probabilities.argmax().item())

            if target_class is None:
                target_class = predicted_class

            score = logits[0, target_class]
            score.backward()

        activations = self.activations
        gradients = self.gradients

        weights = gradients.mean(dim=(2, 3), keepdim=True)

        cam = (
            weights * activations
        ).sum(dim=1)[0]

        cam = torch.relu(cam)

        cam = cam.cpu().numpy()
        cam -= cam.min()

        if cam.max() > 1e-8:
            cam /= cam.max()

        cam = cv2.resize(
            cam,
            (x.shape[-1], x.shape[-2])
        )

        return (
            cam,
            predicted_class,
            probabilities.detach().cpu().numpy()
        )

    def close(self):
        self.handle.remove()


def overlay_cam(image_bgr, cam, alpha=0.45):
    heatmap = cv2.applyColorMap(
        (cam * 255).astype(np.uint8),
        cv2.COLORMAP_JET
    )

    blended = cv2.addWeighted(
        image_bgr, 1-alpha,
        heatmap, alpha, 0
    )

    return cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)


cam_engine = GradCAM(resnet)

print("Grad-CAM 준비 완료!")
print("Target layer: ResNet18 layer4[-1]")

Grad-CAM 준비 완료!
Target layer: ResNet18 layer4[-1]


# 5개 클래스의 Grad-CAM 비교

In [ ]:
RAW_ROOT = Path(RAW_ROOT)

IMAGE_DIR = RAW_ROOT / "images" / "val"
LABEL_DIR = RAW_ROOT / "labels" / "val"

assert IMAGE_DIR.is_dir(), IMAGE_DIR
assert LABEL_DIR.is_dir(), LABEL_DIR

SAMPLES_PER_CLASS = 2
rng = random.Random(42)

samples = {cid: [] for cid in CLASS_NAMES}

image_paths = sorted(
    p for p in IMAGE_DIR.rglob("*")
    if p.suffix.lower() in (".jpg", ".jpeg", ".png")
)

print("Validation images:", len(image_paths))

# GT 라벨에서 클래스별 ROI 후보 수집
for image_path in tqdm(
    image_paths,
    desc="Validation ROI 후보 검색",
    unit="image"
):
    label_path = LABEL_DIR / f"{image_path.stem}.txt"

    if not label_path.is_file():
        continue

    for line in label_path.read_text().splitlines():
        parts = line.split()

        if len(parts) != 5:
            continue

        cid = int(float(parts[0]))
        coords = list(map(float, parts[1:]))

        if cid in samples:
            samples[cid].append((image_path, coords))

selected = []

for cid in CLASS_NAMES:
    candidates = samples[cid]

    if len(candidates) < SAMPLES_PER_CLASS:
        raise RuntimeError(
            f"{CLASS_NAMES[cid]} ROI 부족: {len(candidates)}개"
        )

    picked = rng.sample(candidates, SAMPLES_PER_CLASS)

    for image_path, coords in picked:
        selected.append((cid, image_path, coords))

fig, axes = plt.subplots(
    len(selected), 3,
    figsize=(13, len(selected) * 3),
    constrained_layout=True
)

records = []

for idx, (gt_class, image_path, coords) in enumerate(
    tqdm(selected, desc="Grad-CAM 생성", unit="ROI")
):

    image = cv2.imread(str(image_path))
    if image is None:
        raise RuntimeError(f"이미지 읽기 실패: {image_path}")

    H, W = image.shape[:2]

    cx, cy, bw, bh = coords

    x1 = max(0, int(np.floor((cx - bw/2)*W)))
    y1 = max(0, int(np.floor((cy - bh/2)*H)))
    x2 = min(W, int(np.ceil((cx + bw/2)*W)))
    y2 = min(H, int(np.ceil((cy + bh/2)*H)))

    if x2 <= x1 or y2 <= y1:
        raise ValueError(f"잘못된 GT Box: {image_path}")

    roi = image[y1:y2, x1:x2]

    display_roi, tensor = prepare_roi(roi)

    # 정답 클래스 기준
    gt_cam, pred_class, probabilities = cam_engine.generate(
        tensor,
        target_class=gt_class
    )

    # 예측 클래스 기준
    pred_cam, _, _ = cam_engine.generate(
        tensor,
        target_class=pred_class
    )

    images = [
        cv2.cvtColor(display_roi, cv2.COLOR_BGR2RGB),
        overlay_cam(display_roi, gt_cam),
        overlay_cam(display_roi, pred_cam)
    ]

    titles = [
        f"Original\nGT: {CLASS_NAMES[gt_class]}",
        f"GT class CAM\n{CLASS_NAMES[gt_class]}",
        (
            f"Predicted class CAM\n"
            f"{CLASS_NAMES[pred_class]} "
            f"({probabilities[pred_class]:.1%})"
        )
    ]

    for col in range(3):
        axes[idx, col].imshow(images[col])
        axes[idx, col].set_title(titles[col], fontsize=10)
        axes[idx, col].axis("off")

    records.append({
        "image": image_path.name,
        "gt_class": CLASS_NAMES[gt_class],
        "predicted_class": CLASS_NAMES[pred_class],
        "gt_probability": float(probabilities[gt_class]),
        "pred_probability": float(probabilities[pred_class]),
        "correct": gt_class == pred_class
    })

save_path = CAM_DIR / "gradcam_5classes_preview.png"

fig.savefig(
    save_path,
    dpi=180,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

result_df = pd.DataFrame(records)
display(result_df)

result_df.to_csv(
    CAM_DIR / "gradcam_sample_predictions.csv",
    index=False
)

cam_engine.close()

print("\n완료!")
print("CAM 이미지:", save_path)
print("예측 결과:", CAM_DIR / "gradcam_sample_predictions.csv")

Validation images: 600


Validation ROI 후보 검색:   0%|          | 0/600 [00:00<?, ?image/s]

Grad-CAM 생성:   0%|          | 0/10 [00:00<?, ?ROI/s]

                                               image     gt_class  \
0  styrofoam__1318328@0_05001_221020_P1_T3__0230.jpg    styrofoam   
1  styrofoam__1310862@0_05001_221020_P1_T3__0052.jpg    styrofoam   
2      vinyl__1036136@0_06001_221006_P1_T3__2206.jpg        vinyl   
3      vinyl__1072644@0_06001_221006_P1_T3__1207.jpg        vinyl   
4  brown_glass__809099@0_07001_220822_P1_T3__0605...  brown_glass   
5  brown_glass__809079@0_07001_220822_P1_T3__0691...  brown_glass   
6  green_glass__1095045@0_07002_220919_P1_T3__027...  green_glass   
7  green_glass__1094399@0_07002_220919_P1_T3__018...  green_glass   
8  clear_glass__1130071@0_07003_220919_P1_T3__026...  clear_glass   
9  clear_glass__868743@0_07003_220822_P1_T3__0703...  clear_glass   

  predicted_class  gt_probability  pred_probability  correct  
0       styrofoam        0.992443          0.992443     True  
1       styrofoam        0.999531          0.999531     True  
2           vinyl        0.989700          0.989700


완료!
CAM 이미지: /content/drive/MyDrive/Colab Notebooks/AI개론/261110스터디/multiclass_yolo26n/experiments_v3/cam_analysis/gradcam_5classes_preview.png
예측 결과: /content/drive/MyDrive/Colab Notebooks/AI개론/261110스터디/multiclass_yolo26n/experiments_v3/cam_analysis/gradcam_sample_predictions.csv


In [ ]:

from pathlib import Path
import shutil

# 기존 TAR 및 학습 모델 경로는 그대로 유지
# BASE, MODEL_DIR, RAW_ROOT 수정하지 않음!

NOTEBOOK_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/AI응용"
)

assert NOTEBOOK_DIR.is_dir(), (
    f"AI응용 폴더를 찾을 수 없습니다: {NOTEBOOK_DIR}"
)

# 새로운 실험 결과 경로
EXPERIMENT_DIR = (
    NOTEBOOK_DIR
    / "261113_zerowaste_reuse_experiments"
    / "experiments_v3"
)

NEW_CAM_DIR = EXPERIMENT_DIR / "cam_analysis"
NEW_CAM_DIR.mkdir(parents=True, exist_ok=True)

# 이미 생성했던 산출물은 새 폴더로 복사
OLD_CAM_DIR = Path(CAM_DIR)

for filename in [
    "gradcam_5classes_preview.png",
    "gradcam_sample_predictions.csv"
]:
    src = OLD_CAM_DIR / filename
    dst = NEW_CAM_DIR / filename

    if src.is_file() and src.resolve() != dst.resolve():
        shutil.copy2(src, dst)
        print("복사 완료:", dst)

# 이후 모든 CAM 결과는 새 위치에 저장
CAM_DIR = NEW_CAM_DIR

# 후속 실험별 결과 폴더
for dirname in [
    "pseudo_masks",
    "augmentation_ablation",
    "dropout_ablation",
    "evaluation"
]:
    (EXPERIMENT_DIR / dirname).mkdir(
        parents=True,
        exist_ok=True
    )

print("\n✅ 저장 경로 변경 완료!")
print("입력 데이터:", BASE)
print("출력 결과:", EXPERIMENT_DIR)
print("CAM 결과:", CAM_DIR)


복사 완료: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/cam_analysis/gradcam_5classes_preview.png
복사 완료: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/cam_analysis/gradcam_sample_predictions.csv

✅ 저장 경로 변경 완료!
입력 데이터: /content/drive/MyDrive/Colab Notebooks/AI개론/261110스터디/multiclass_yolo26n
출력 결과: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3
CAM 결과: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/cam_analysis


# Validation 전체 오류 탐색 + Grad-CAM

In [ ]:

import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# ============================================================
# 0. 환경 확인
# ============================================================

for name in [
    "RAW_ROOT",
    "CAM_DIR",
    "resnet",
    "prepare_roi",
    "overlay_cam",
    "GradCAM",
    "CLASS_NAMES",
    "DEVICE"
]:
    assert name in globals(), f"이전 셀의 {name} 변수가 없습니다."

RAW_ROOT = Path(RAW_ROOT)
CAM_DIR = Path(CAM_DIR)
CAM_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_DIR = RAW_ROOT / "images" / "val"
LABEL_DIR = RAW_ROOT / "labels" / "val"

assert IMAGE_DIR.is_dir(), IMAGE_DIR
assert LABEL_DIR.is_dir(), LABEL_DIR

resnet.eval()

# 선택할 어려운 사례 수
MISCLASSIFIED_PER_CLASS = 3
LOW_CONFIDENCE_PER_CLASS = 2

class_ids = sorted(CLASS_NAMES.keys())
class_labels = [CLASS_NAMES[c] for c in class_ids]

image_paths = sorted(
    p for p in IMAGE_DIR.rglob("*")
    if p.suffix.lower() in (".jpg", ".jpeg", ".png")
)

print("Validation 이미지 수:", len(image_paths))
print("결과 저장 경로:", CAM_DIR)

# ============================================================
# 1. Validation 전체 GT ROI 분류
# ============================================================

print("\n[1/5] Validation GT ROI 전체 분류")

records = []
skipped_images = 0
skipped_boxes = 0

for image_path in tqdm(
    image_paths,
    desc="1/5 Validation 분류",
    unit="image"
):

    # 하위 디렉터리가 있어도 대응 가능
    relative_path = image_path.relative_to(IMAGE_DIR)

    label_path = (
        LABEL_DIR / relative_path
    ).with_suffix(".txt")

    if not label_path.exists():
        continue

    image = cv2.imread(str(image_path))

    if image is None:
        skipped_images += 1
        continue

    H, W = image.shape[:2]

    lines = label_path.read_text().splitlines()

    for object_idx, line in enumerate(lines):

        values = line.strip().split()

        if len(values) != 5:
            continue

        gt_class = int(float(values[0]))

        if gt_class not in CLASS_NAMES:
            continue

        cx, cy, bw, bh = map(float, values[1:])

        x1 = max(0, int(np.floor((cx - bw / 2) * W)))
        y1 = max(0, int(np.floor((cy - bh / 2) * H)))
        x2 = min(W, int(np.ceil((cx + bw / 2) * W)))
        y2 = min(H, int(np.ceil((cy + bh / 2) * H)))

        if x2 <= x1 or y2 <= y1:
            skipped_boxes += 1
            continue

        roi = image[y1:y2, x1:x2]

        if roi.size == 0:
            skipped_boxes += 1
            continue

        _, tensor = prepare_roi(roi)

        with torch.inference_mode():

            logits = resnet(tensor)["class_id"]

            probs = torch.softmax(
                logits, dim=1
            )[0].cpu().numpy()

        pred_class = int(np.argmax(probs))

        records.append({
            "image": image_path.name,
            "image_path": str(image_path),
            "object_idx": object_idx,

            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,

            "gt_id": gt_class,
            "pred_id": pred_class,

            "gt_class": CLASS_NAMES[gt_class],
            "pred_class": CLASS_NAMES[pred_class],

            "gt_probability": float(probs[gt_class]),
            "pred_probability": float(probs[pred_class]),

            "correct": gt_class == pred_class,

            **{
                f"prob_{CLASS_NAMES[c]}": float(probs[c])
                for c in class_ids
            }
        })

df = pd.DataFrame(records)

assert len(df) > 0, "평가 가능한 GT ROI가 없습니다."

predictions_path = (
    CAM_DIR / "validation_all_roi_predictions.csv"
)

df.to_csv(predictions_path, index=False)

print("\n전체 GT ROI:", len(df))
print("정분류:", int(df["correct"].sum()))
print("오분류:", int((~df["correct"]).sum()))
print("이미지 읽기 실패:", skipped_images)
print("유효하지 않은 Box:", skipped_boxes)

# ============================================================
# 2. 전체 정확도, Macro-F1, 혼동행렬
# ============================================================

print("\n[2/5] 클래스별 성능 계산")

y_true = df["gt_id"].to_numpy()
y_pred = df["pred_id"].to_numpy()

accuracy = accuracy_score(y_true, y_pred)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=class_ids,
    average="macro",
    zero_division=0
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Macro-F1 : {macro_f1:.4f}")

print("\nClassification Report")

print(
    classification_report(
        y_true,
        y_pred,
        labels=class_ids,
        target_names=class_labels,
        digits=4,
        zero_division=0
    )
)

report = classification_report(
    y_true,
    y_pred,
    labels=class_ids,
    target_names=class_labels,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T

report_df.to_csv(
    CAM_DIR / "validation_classification_report.csv"
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=class_ids
)

cm_df = pd.DataFrame(
    cm,
    index=class_labels,
    columns=class_labels
)

display(cm_df)

cm_df.to_csv(
    CAM_DIR / "validation_confusion_matrix.csv"
)

# 혼동행렬 시각화
fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(cm, cmap="Blues")

ax.set_xticks(range(len(class_labels)))
ax.set_yticks(range(len(class_labels)))

ax.set_xticklabels(class_labels, rotation=35, ha="right")
ax.set_yticklabels(class_labels)

ax.set_xlabel("Predicted class")
ax.set_ylabel("Ground truth class")

ax.set_title(
    "Validation GT ROI Classification Confusion Matrix"
)

for i in range(len(class_labels)):
    for j in range(len(class_labels)):

        ax.text(
            j, i, str(cm[i, j]),
            ha="center",
            va="center",
            color=(
                "white"
                if cm[i, j] > cm.max() / 2
                else "black"
            )
        )

fig.colorbar(im, ax=ax)
plt.tight_layout()

cm_path = CAM_DIR / "validation_confusion_matrix.png"

fig.savefig(cm_path, dpi=180, bbox_inches="tight")

plt.show()
plt.close(fig)

# ============================================================
# 3. 오분류 + 확신이 낮은 사례 추출
# ============================================================

print("\n[3/5] 어려운 사례 자동 선정")

selected_frames = []

for cid in class_ids:

    subset = df[df["gt_id"] == cid]

    # A. 오분류 사례: 정답 확률이 낮은 순서
    incorrect = (
        subset[~subset["correct"]]
        .sort_values("gt_probability", ascending=True)
        .head(MISCLASSIFIED_PER_CLASS)
        .copy()
    )

    incorrect["case_type"] = "misclassified"

    # B. 정분류 사례 중 확신이 낮은 순서
    uncertain = (
        subset[subset["correct"]]
        .sort_values("pred_probability", ascending=True)
        .head(LOW_CONFIDENCE_PER_CLASS)
        .copy()
    )

    uncertain["case_type"] = "low_confidence"

    selected_frames.extend([incorrect, uncertain])

hard_cases = pd.concat(
    selected_frames,
    ignore_index=True
)

hard_cases_path = CAM_DIR / "validation_hard_cases.csv"

hard_cases.to_csv(
    hard_cases_path,
    index=False
)

print("전체 오분류 수:", int((~df["correct"]).sum()))
print("Grad-CAM 분석 대상:", len(hard_cases))

display(
    hard_cases[
        [
            "gt_class",
            "pred_class",
            "gt_probability",
            "pred_probability",
            "case_type",
            "image"
        ]
    ]
)

# ============================================================
# 4. 어려운 사례의 Grad-CAM 생성
# ============================================================

print("\n[4/5] 어려운 사례 Grad-CAM 생성")

# 이전 셀에서 close()했으므로 새로운 Hook 등록
cam_engine_hard = GradCAM(resnet)

saved_paths = []

try:

    for cid in tqdm(
        class_ids,
        desc="4/5 클래스별 Grad-CAM",
        unit="class"
    ):

        cases = hard_cases[
            hard_cases["gt_id"] == cid
        ].reset_index(drop=True)

        if cases.empty:
            continue

        fig, axes = plt.subplots(
            len(cases), 3,
            figsize=(13, 3.8 * len(cases)),
            squeeze=False,
            constrained_layout=True
        )

        for row_idx, case in cases.iterrows():

            image = cv2.imread(
                str(case["image_path"])
            )

            if image is None:
                continue

            x1, y1, x2, y2 = (
                int(case[k])
                for k in ["x1", "y1", "x2", "y2"]
            )

            roi = image[y1:y2, x1:x2]

            if roi.size == 0:
                continue

            display_roi, tensor = prepare_roi(roi)

            gt_class = int(case["gt_id"])
            pred_class = int(case["pred_id"])

            # 정답 클래스의 Grad-CAM
            gt_cam, _, _ = cam_engine_hard.generate(
                tensor,
                target_class=gt_class
            )

            # 예측 클래스가 다를 때만 별도 계산
            if gt_class == pred_class:
                pred_cam = gt_cam.copy()
            else:
                pred_cam, _, _ = cam_engine_hard.generate(
                    tensor,
                    target_class=pred_class
                )

            images = [
                cv2.cvtColor(
                    display_roi,
                    cv2.COLOR_BGR2RGB
                ),
                overlay_cam(display_roi, gt_cam),
                overlay_cam(display_roi, pred_cam)
            ]

            titles = [
                (
                    f"Original [{case['case_type']}]\n"
                    f"GT: {case['gt_class']}"
                ),
                (
                    f"GT CAM: {case['gt_class']}\n"
                    f"P(GT)={case['gt_probability']:.1%}"
                ),
                (
                    f"Pred CAM: {case['pred_class']}\n"
                    f"P(Pred)={case['pred_probability']:.1%}"
                )
            ]

            for col in range(3):

                ax = axes[row_idx, col]

                ax.imshow(images[col])

                ax.set_title(
                    titles[col],
                    fontsize=10
                )

                ax.axis("off")

        class_name = CLASS_NAMES[cid]

        fig.suptitle(
            f"Hard Cases Grad-CAM - {class_name}",
            fontsize=15
        )

        save_path = (
            CAM_DIR /
            f"gradcam_hard_cases_{class_name}.png"
        )

        fig.savefig(
            save_path,
            dpi=160,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(fig)

        saved_paths.append(str(save_path))

finally:
    cam_engine_hard.close()

# ============================================================
# 5. 클래스별 오류 통계
# ============================================================

print("\n[5/5] 클래스별 오류 통계")

summary = (
    df.groupby("gt_class", sort=False)
      .agg(
          total=("correct", "size"),
          correct=("correct", "sum"),
          avg_gt_probability=("gt_probability", "mean")
      )
      .reset_index()
)

summary["incorrect"] = (
    summary["total"] - summary["correct"]
)

summary["accuracy"] = (
    summary["correct"] / summary["total"]
)

summary = summary[
    [
        "gt_class",
        "total",
        "correct",
        "incorrect",
        "accuracy",
        "avg_gt_probability"
    ]
]

display(summary)

summary.to_csv(
    CAM_DIR / "validation_class_error_summary.csv",
    index=False
)

print("\n✅ Validation 전체 오류 분석 완료!")
print("Accuracy:", f"{accuracy:.2%}")
print("Macro-F1:", f"{macro_f1:.4f}")

print("\n결과 저장 위치:")
print(CAM_DIR)

print("\n저장된 Grad-CAM 이미지:")
for p in saved_paths:
    print(p)

print("\nCSV:")
print(predictions_path)
print(hard_cases_path)


Validation 이미지 수: 600
결과 저장 경로: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/cam_analysis

[1/5] Validation GT ROI 전체 분류


1/5 Validation 분류:   0%|          | 0/600 [00:00<?, ?image/s]


전체 GT ROI: 1365
정분류: 1290
오분류: 75
이미지 읽기 실패: 0
유효하지 않은 Box: 0

[2/5] 클래스별 성능 계산
Accuracy : 0.9451
Macro-F1 : 0.9336

Classification Report
              precision    recall  f1-score   support

   styrofoam     0.9861    0.9779    0.9820       362
       vinyl     0.9326    0.9907    0.9607       321
 brown_glass     0.9647    0.9436    0.9540       319
 green_glass     0.8342    0.9306    0.8798       173
 clear_glass     0.9750    0.8211    0.8914       190

    accuracy                         0.9451      1365
   macro avg     0.9385    0.9328    0.9336      1365
weighted avg     0.9477    0.9451    0.9449      1365



             styrofoam  vinyl  brown_glass  green_glass  clear_glass
styrofoam          354      7            1            0            0
vinyl                3    318            0            0            0
brown_glass          0      4          301           13            1
green_glass          0      4            5          161            3
clear_glass          2      8            5           19          156


[3/5] 어려운 사례 자동 선정
전체 오분류 수: 75
Grad-CAM 분석 대상: 25


       gt_class   pred_class  gt_probability  pred_probability  \
0     styrofoam        vinyl        0.071772          0.910189   
1     styrofoam        vinyl        0.073167          0.925216   
2     styrofoam        vinyl        0.074167          0.922613   
3     styrofoam    styrofoam        0.518990          0.518990   
4     styrofoam    styrofoam        0.573549          0.573549   
5         vinyl    styrofoam        0.278741          0.701837   
6         vinyl    styrofoam        0.340490          0.650108   
7         vinyl    styrofoam        0.484592          0.495604   
8         vinyl        vinyl        0.506758          0.506758   
9         vinyl        vinyl        0.588066          0.588066   
10  brown_glass  green_glass        0.000284          0.999249   
11  brown_glass        vinyl        0.000835          0.996414   
12  brown_glass  green_glass        0.003681          0.996306   
13  brown_glass  brown_glass        0.551138          0.551138   
14  brown_


[4/5] 어려운 사례 Grad-CAM 생성


4/5 클래스별 Grad-CAM:   0%|          | 0/5 [00:00<?, ?class/s]


[5/5] 클래스별 오류 통계


      gt_class  total  correct  incorrect  accuracy  avg_gt_probability
0  brown_glass    319      301         18  0.943574            0.930775
1  clear_glass    190      156         34  0.821053            0.795956
2  green_glass    173      161         12  0.930636            0.884799
3    styrofoam    362      354          8  0.977901            0.966200
4        vinyl    321      318          3  0.990654            0.979415


✅ Validation 전체 오류 분석 완료!
Accuracy: 94.51%
Macro-F1: 0.9336

결과 저장 위치:
/content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/cam_analysis

저장된 Grad-CAM 이미지:
/content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/cam_analysis/gradcam_hard_cases_styrofoam.png
/content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/cam_analysis/gradcam_hard_cases_vinyl.png
/content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/cam_analysis/gradcam_hard_cases_brown_glass.png
/content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/cam_analysis/gradcam_hard_cases_green_glass.png
/content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/cam_analysis/gradcam_hard_cases_clear_glass.png

CSV:
/content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/

# 오분류 원본 이미지와 GT Box 검수

In [ ]:

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm

# ============================================================
# 1. 검수 환경
# ============================================================

AUDIT_DIR = Path(CAM_DIR).parent / "annotation_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

ROWS_PER_SHEET = 5
MAX_REVIEW = 30  # 우선 오분류 30건 검수, 전체는 None

# 오분류 사례만 추출
errors = df.loc[~df["correct"]].copy()

# 유리병 문제를 먼저 살펴봄
priority = {
    2: 0,  # Brown
    3: 1,  # Green
    4: 2,  # Clear
    0: 3,  # Styrofoam
    1: 4   # Vinyl
}

errors["priority"] = errors["gt_id"].map(priority)

errors = errors.sort_values(
    ["priority", "pred_probability"],
    ascending=[True, False]
).reset_index(drop=True)

if MAX_REVIEW is not None:
    errors = errors.head(MAX_REVIEW).copy()

# 수동 검수용 열
errors["review_status"] = "pending"
errors["visual_class"] = ""
errors["review_note"] = ""

csv_path = AUDIT_DIR / "annotation_review.csv"
errors.to_csv(csv_path, index=False)

# ============================================================
# 2. 원본 이미지의 모든 GT Box 표시
# ============================================================

def draw_gt_boxes(image, image_path, selected_box):

    canvas = image.copy()

    image_path = Path(image_path)
    relative_path = image_path.relative_to(IMAGE_DIR)

    label_path = (
        LABEL_DIR / relative_path
    ).with_suffix(".txt")

    H, W = image.shape[:2]

    if label_path.exists():
        for line in label_path.read_text().splitlines():

            parts = line.split()
            if len(parts) != 5:
                continue

            cid = int(float(parts[0]))
            cx, cy, bw, bh = map(float, parts[1:])

            xx1 = int((cx - bw/2) * W)
            yy1 = int((cy - bh/2) * H)
            xx2 = int((cx + bw/2) * W)
            yy2 = int((cy + bh/2) * H)

            cv2.rectangle(
                canvas,
                (xx1, yy1),
                (xx2, yy2),
                (255, 255, 0),  # BGR: cyan
                2
            )

    x1, y1, x2, y2 = selected_box

    cv2.rectangle(
        canvas,
        (x1, y1),
        (x2, y2),
        (0, 255, 255),  # BGR: yellow
        3
    )

    return canvas

# ============================================================
# 3. 원본 전체 이미지 + ROI 시각화
# ============================================================

print("검수 대상:", len(errors))

for page_start in tqdm(
    range(0, len(errors), ROWS_PER_SHEET),
    desc="Annotation Audit",
    unit="sheet"
):

    batch = errors.iloc[
        page_start:page_start + ROWS_PER_SHEET
    ]

    fig, axes = plt.subplots(
        len(batch), 2,
        figsize=(14, 4 * len(batch)),
        squeeze=False,
        constrained_layout=True
    )

    for row_idx, (_, case) in enumerate(batch.iterrows()):

        image = cv2.imread(str(case["image_path"]))

        if image is None:
            continue

        box = tuple(
            int(case[k])
            for k in ("x1", "y1", "x2", "y2")
        )

        x1, y1, x2, y2 = box

        annotated = draw_gt_boxes(
            image, case["image_path"], box
        )

        roi = image[y1:y2, x1:x2]

        axes[row_idx, 0].imshow(
            cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        )

        axes[row_idx, 0].set_title(
            f"Full image | Object #{int(case['object_idx'])}\n"
            f"{case['image']}",
            fontsize=9
        )

        axes[row_idx, 1].imshow(
            cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
        )

        axes[row_idx, 1].set_title(
            f"GT: {case['gt_class']}\n"
            f"Pred: {case['pred_class']} "
            f"({case['pred_probability']:.1%})",
            fontsize=10
        )

        for ax in axes[row_idx]:
            ax.axis("off")

    page_num = page_start // ROWS_PER_SHEET + 1

    output_path = (
        AUDIT_DIR / f"annotation_audit_{page_num:02d}.png"
    )

    fig.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

print("\n검수 이미지 저장:", AUDIT_DIR)
print("검수용 CSV:", csv_path)


검수 대상: 30


Annotation Audit:   0%|          | 0/6 [00:00<?, ?sheet/s]


검수 이미지 저장: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/annotation_audit
검수용 CSV: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/annotation_audit/annotation_review.csv


# RAW vs Kornia, 동일 1,365개 ROI 비교

In [ ]:

import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 0. 기본 설정 및 데이터 검증
# ============================================================

print("[1/5] 실험 환경 확인")

REQUIRED = [
    "df", "MODEL_DIR", "CLASS_NAMES",
    "MultiTaskResNet18", "prepare_roi", "DEVICE"
]

for name in REQUIRED:
    assert name in globals(), f"기존 변수 없음: {name}"

# 이전 셀에서 변경한 AI응용 저장 경로 재사용
OUTPUT_DIR = (
    Path(CAM_DIR).parent
    / "evaluation"
    / "raw_vs_kornia_same_roi"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATHS = {
    "RAW": (
        Path(MODEL_DIR)
        / "resnet18_multitask_raw_roi7579_v1_best.pt"
    ),
    "Kornia": (
        Path(MODEL_DIR)
        / "resnet18_multitask_kornia_gpu_roi7579_v1_best.pt"
    )
}

for name, path in MODEL_PATHS.items():
    assert path.is_file(), f"{name} 모델 없음: {path}"

# 셀 6에서 평가한 1,365개 GT ROI를 그대로 사용
eval_df = df.reset_index(drop=True).copy()

required_columns = [
    "image_path", "object_idx",
    "x1", "y1", "x2", "y2", "gt_id"
]

assert all(c in eval_df.columns for c in required_columns)
assert len(eval_df) == 1365, (
    f"예상 ROI 1365개, 현재 {len(eval_df)}개"
)

assert not eval_df.duplicated(
    subset=["image_path", "object_idx"]
).any(), "ROI 식별자가 중복됐습니다."

class_ids = sorted(CLASS_NAMES.keys())
class_names = [CLASS_NAMES[c] for c in class_ids]

assert eval_df["gt_id"].isin(class_ids).all()

y_true = eval_df["gt_id"].to_numpy(dtype=int)

print("Validation GT ROI:", len(eval_df))
print("원본 이미지:", eval_df["image_path"].nunique())
print("Device:", DEVICE)
print("출력:", OUTPUT_DIR)

# ============================================================
# 1. 기존 PyTorch 모델 2개 로드
# ============================================================

print("\n[2/5] 체크포인트 로드")

models = {}

for name, path in tqdm(
    MODEL_PATHS.items(),
    desc="2/5 모델 로드",
    total=len(MODEL_PATHS),
    unit="model"
):
    # 본인이 저장한 신뢰할 수 있는 체크포인트만 로드
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    state = (
        checkpoint["model_state"]
        if isinstance(checkpoint, dict)
        and "model_state" in checkpoint
        else checkpoint
    )

    model = MultiTaskResNet18(dropout_p=0.2)

    model.load_state_dict(state, strict=True)

    model = model.to(DEVICE)
    model.eval()

    models[name] = model

    if isinstance(checkpoint, dict):
        print(
            name,
            "| Epoch:",
            checkpoint.get("epoch", "unknown")
        )

print("두 모델 로드 완료")

# ============================================================
# 2. 동일 ROI를 한 번 전처리하여 두 모델에 입력
# ============================================================

print("\n[3/5] 동일 Validation ROI 예측")

BATCH_SIZE = 32

prob_chunks = {
    "RAW": [],
    "Kornia": []
}

batch_tensors = []

def run_shared_batch(tensors):
    """동일한 텐서 배치를 두 체크포인트에 입력."""

    x = torch.stack(tensors, dim=0).to(DEVICE)

    with torch.inference_mode():
        for name, model in models.items():
            logits = model(x)["class_id"]

            probabilities = torch.softmax(
                logits, dim=1
            )

            prob_chunks[name].append(
                probabilities.cpu().numpy()
            )

for row in tqdm(
    eval_df.itertuples(index=False),
    total=len(eval_df),
    desc="3/5 GT ROI 비교",
    unit="ROI"
):

    image_path = Path(row.image_path)
    image = cv2.imread(str(image_path))

    if image is None:
        raise RuntimeError(
            f"이미지 읽기 실패: {image_path}"
        )

    h, w = image.shape[:2]

    x1 = int(row.x1)
    y1 = int(row.y1)
    x2 = int(row.x2)
    y2 = int(row.y2)

    assert 0 <= x1 < x2 <= w, (image_path, x1, x2)
    assert 0 <= y1 < y2 <= h, (image_path, y1, y2)

    roi = image[y1:y2, x1:x2]

    # 셀 6에서 사용했던 전처리를 동일하게 적용
    # Letterbox + JPEG round-trip + Normalize
    _, tensor = prepare_roi(roi)

    batch_tensors.append(tensor.squeeze(0))

    if len(batch_tensors) >= BATCH_SIZE:
        run_shared_batch(batch_tensors)
        batch_tensors.clear()

if batch_tensors:
    run_shared_batch(batch_tensors)
    batch_tensors.clear()

probabilities = {
    name: np.concatenate(chunks, axis=0)
    for name, chunks in prob_chunks.items()
}

for name, probs in probabilities.items():
    assert probs.shape == (
        len(eval_df), len(class_ids)
    ), (name, probs.shape)

print("동일 전처리 비교 완료")

# ============================================================
# 3. Accuracy, Macro-F1, 클래스별 F1 계산
# ============================================================

print("\n[4/5] F1 및 혼동행렬 비교")

paired = eval_df[
    [
        "image", "image_path", "object_idx",
        "x1", "y1", "x2", "y2",
        "gt_id", "gt_class"
    ]
].copy()

metrics = {}
class_reports = {}
confusion = {}

for name, probs in probabilities.items():

    preds = np.argmax(probs, axis=1)

    paired[f"{name}_pred_id"] = preds

    paired[f"{name}_pred_class"] = [
        CLASS_NAMES[int(p)] for p in preds
    ]

    paired[f"{name}_confidence"] = probs.max(axis=1)

    paired[f"{name}_gt_probability"] = probs[
        np.arange(len(y_true)),
        y_true
    ]

    paired[f"{name}_correct"] = preds == y_true

    accuracy = accuracy_score(y_true, preds)

    macro_f1 = f1_score(
        y_true,
        preds,
        labels=class_ids,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        preds,
        labels=class_ids,
        average="weighted",
        zero_division=0
    )

    metrics[name] = {
        "model": name,
        "num_roi": len(y_true),
        "correct": int((preds == y_true).sum()),
        "incorrect": int((preds != y_true).sum()),
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

    class_reports[name] = classification_report(
        y_true,
        preds,
        labels=class_ids,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    confusion[name] = confusion_matrix(
        y_true,
        preds,
        labels=class_ids
    )

summary_df = pd.DataFrame([
    metrics["RAW"],
    metrics["Kornia"]
])

display(summary_df)

summary_df.to_csv(
    OUTPUT_DIR / "model_comparison_summary.csv",
    index=False
)

# 클래스별 비교
class_rows = []

for cid in class_ids:

    cname = CLASS_NAMES[cid]

    raw_result = class_reports["RAW"][cname]
    kor_result = class_reports["Kornia"][cname]

    class_rows.append({
        "class": cname,
        "support": int(raw_result["support"]),
        "RAW_precision": raw_result["precision"],
        "Kornia_precision": kor_result["precision"],
        "RAW_recall": raw_result["recall"],
        "Kornia_recall": kor_result["recall"],
        "RAW_F1": raw_result["f1-score"],
        "Kornia_F1": kor_result["f1-score"],
        "F1_delta_pp": (
            kor_result["f1-score"]
            - raw_result["f1-score"]
        ) * 100
    })

class_df = pd.DataFrame(class_rows)

display(class_df)

class_df.to_csv(
    OUTPUT_DIR / "model_comparison_per_class.csv",
    index=False
)

# F1 비교 그래프
fig, ax = plt.subplots(figsize=(10, 5))

positions = np.arange(len(class_names))
bar_width = 0.36

ax.bar(
    positions - bar_width / 2,
    class_df["RAW_F1"] * 100,
    bar_width,
    label="RAW"
)

ax.bar(
    positions + bar_width / 2,
    class_df["Kornia_F1"] * 100,
    bar_width,
    label="Kornia"
)

ax.set_xticks(positions)
ax.set_xticklabels(class_names, rotation=25, ha="right")
ax.set_ylabel("F1 (%)")
ax.set_ylim(0, 105)

ax.set_title(
    "Class-wise F1: RAW vs Kornia\n"
    "Same 1365 Validation GT ROIs"
)

ax.legend()
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "class_f1_comparison.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# 같은 색상 범위로 혼동행렬 비교
fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6),
    constrained_layout=True
)

max_count = max(
    cm.max() for cm in confusion.values()
)

for ax, name in zip(axes, ["RAW", "Kornia"]):

    cm = confusion[name]

    im = ax.imshow(
        cm,
        cmap="Blues",
        vmin=0,
        vmax=max_count
    )

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))

    ax.set_xticklabels(
        class_names, rotation=45, ha="right"
    )

    ax.set_yticklabels(class_names)

    ax.set_xlabel("Predicted")
    ax.set_ylabel("GT")

    ax.set_title(
        f"{name}\nMacro-F1={metrics[name]['macro_f1']:.4f}"
    )

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            value = int(cm[i, j])

            ax.text(
                j, i, str(value),
                ha="center",
                va="center",
                fontsize=10,
                color=(
                    "white"
                    if value > max_count * 0.5
                    else "black"
                )
            )

fig.colorbar(
    im,
    ax=axes,
    shrink=0.75,
    label="ROI count"
)

fig.savefig(
    OUTPUT_DIR / "confusion_raw_vs_kornia.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ============================================================
# 4. 같은 ROI에서 두 모델의 예측 차이 비교
# ============================================================

print("\n[5/5] 두 모델의 차이 분석")

raw_ok = paired["RAW_correct"].to_numpy()
kor_ok = paired["Kornia_correct"].to_numpy()

paired["comparison"] = np.select(
    [
        raw_ok & kor_ok,
        raw_ok & ~kor_ok,
        ~raw_ok & kor_ok
    ],
    [
        "both_correct",
        "RAW_only_correct",
        "Kornia_only_correct"
    ],
    default="both_wrong"
)

paired["prediction_changed"] = (
    paired["RAW_pred_id"]
    != paired["Kornia_pred_id"]
)

paired.to_csv(
    OUTPUT_DIR / "paired_predictions_1365.csv",
    index=False
)

changed = paired[
    paired["prediction_changed"]
].copy()

changed.to_csv(
    OUTPUT_DIR / "prediction_disagreements.csv",
    index=False
)

# 예측의 정오 여부가 달라진 경우만 추출
correctness_changed = paired[
    paired["RAW_correct"] != paired["Kornia_correct"]
].copy()

correctness_changed.to_csv(
    OUTPUT_DIR / "correctness_changed.csv",
    index=False
)

comparison_counts = (
    paired["comparison"]
    .value_counts()
    .reindex(
        [
            "both_correct",
            "RAW_only_correct",
            "Kornia_only_correct",
            "both_wrong"
        ],
        fill_value=0
    )
)

print("\n동일 ROI 기준 결과")
display(comparison_counts.to_frame("ROI_count"))

raw_f1 = metrics["RAW"]["macro_f1"]
kor_f1 = metrics["Kornia"]["macro_f1"]

f1_delta_pp = (kor_f1 - raw_f1) * 100

print("\n========== 최종 비교 ==========")
print(f"RAW Macro-F1    : {raw_f1:.4f} ({raw_f1:.2%})")
print(f"Kornia Macro-F1 : {kor_f1:.4f} ({kor_f1:.2%})")
print(f"F1 차이         : {f1_delta_pp:+.3f}%p")

print(
    "Accuracy 차이   :",
    f"{(metrics['Kornia']['accuracy'] - metrics['RAW']['accuracy']) * 100:+.3f}%p"
)

print(
    "RAW만 정답      :",
    int((paired["comparison"] == "RAW_only_correct").sum())
)

print(
    "Kornia만 정답   :",
    int((paired["comparison"] == "Kornia_only_correct").sum())
)

print("예측 클래스 불일치:", len(changed))

# 기존 셀 6의 RAW 결과와 재현성 비교
previous_raw_f1 = 0.9336

if abs(raw_f1 - previous_raw_f1) > 0.001:
    print(
        "\n⚠️ RAW F1이 앞선 0.9336과 차이 납니다."
        "\n전처리, ROI 순서, 모델 체크포인트를 확인하세요."
    )
else:
    print("\n✅ 기존 RAW 평가와 거의 일치합니다.")

print("\n저장 경로:", OUTPUT_DIR)
print("완료!")


[1/5] 실험 환경 확인
Validation GT ROI: 1365
원본 이미지: 600
Device: cuda
출력: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/evaluation/raw_vs_kornia_same_roi

[2/5] 체크포인트 로드


2/5 모델 로드:   0%|          | 0/2 [00:00<?, ?model/s]

RAW | Epoch: 2
Kornia | Epoch: 14
두 모델 로드 완료

[3/5] 동일 Validation ROI 예측


3/5 GT ROI 비교:   0%|          | 0/1365 [00:00<?, ?ROI/s]

동일 전처리 비교 완료

[4/5] F1 및 혼동행렬 비교


    model  num_roi  correct  incorrect  accuracy  macro_f1  weighted_f1
0     RAW     1365     1290         75  0.945055  0.933589     0.944893
1  Kornia     1365     1311         54  0.960440  0.951525     0.960541

         class  support  RAW_precision  Kornia_precision  RAW_recall  \
0    styrofoam      362       0.986072          0.991667    0.977901   
1        vinyl      321       0.932551          0.972393    0.990654   
2  brown_glass      319       0.964744          0.974277    0.943574   
3  green_glass      173       0.834197          0.888268    0.930636   
4  clear_glass      190       0.975000          0.925926    0.821053   

   Kornia_recall    RAW_F1  Kornia_F1  F1_delta_pp  
0       0.986188  0.981969   0.988920     0.695018  
1       0.987539  0.960725   0.979907     1.918219  
2       0.949843  0.954041   0.961905     0.786356  
3       0.919075  0.879781   0.903409     2.362767  
4       0.921053  0.891429   0.923483     3.205428  


[5/5] 두 모델의 차이 분석

동일 ROI 기준 결과


                     ROI_count
comparison                    
both_correct              1270
RAW_only_correct            20
Kornia_only_correct         41
both_wrong                  34


========== 최종 비교 ==========
RAW Macro-F1    : 0.9336 (93.36%)
Kornia Macro-F1 : 0.9515 (95.15%)
F1 차이         : +1.794%p
Accuracy 차이   : +1.538%p
RAW만 정답      : 20
Kornia만 정답   : 41
예측 클래스 불일치: 63

✅ 기존 RAW 평가와 거의 일치합니다.

저장 경로: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/evaluation/raw_vs_kornia_same_roi
완료!


# YOLO26n 실제 증강 학습 실험

In [ ]:

# ============================================================
# 셀 9. YOLO26n Brightness / HSV Ablation
# 기존 RAW best.pt에서 동일 조건으로 추가 학습
# ============================================================

from pathlib import Path
from ultralytics import YOLO
from tqdm.auto import tqdm
import torch
import numpy as np
import pandas as pd
import yaml

assert torch.cuda.is_available(), "Colab GPU 런타임이 필요합니다."

# 기존 산출물: 읽기 전용
DATA_ROOT = Path(RAW_ROOT)
START_WEIGHTS = (
    Path(MODEL_DIR) / "yolo26n_raw_retrain_best.pt"
)

assert START_WEIGHTS.is_file(), START_WEIGHTS

for split in ("train", "val", "test"):
    assert (DATA_ROOT / "images" / split).is_dir()
    assert (DATA_ROOT / "labels" / split).is_dir()

# 새 산출물: AI응용 폴더
EXP_ROOT = (
    Path(CAM_DIR).parent / "augmentation_ablation"
)
EXP_ROOT.mkdir(parents=True, exist_ok=True)

CLASS_NAMES_YOLO = [
    "styrofoam",
    "vinyl",
    "brown_glass",
    "green_glass",
    "clear_glass"
]

# 기존 Train/Val/Test 분할 그대로 사용
DATA_YAML = EXP_ROOT / "dataset.yaml"

data_config = {
    "path": str(DATA_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 5,
    "names": CLASS_NAMES_YOLO
}

with DATA_YAML.open("w", encoding="utf-8") as f:
    yaml.safe_dump(
        data_config, f,
        allow_unicode=True,
        sort_keys=False
    )

# 변경할 요소는 HSV 증강뿐
EXPERIMENTS = {
    "control": {
        "hsv_h": 0.0,
        "hsv_s": 0.0,
        "hsv_v": 0.0
    },
    "brightness": {
        "hsv_h": 0.0,
        "hsv_s": 0.0,
        "hsv_v": 0.15
    },
    "mild_hsv": {
        "hsv_h": 0.005,
        "hsv_s": 0.10,
        "hsv_v": 0.15
    }
}

EPOCHS = 12
BATCH = 16
IMGSZ = 640
SEED = 42

summary_rows = []
class_rows = []

print("시작 체크포인트:", START_WEIGHTS)
print("학습 데이터:", DATA_ROOT)
print("결과 저장:", EXP_ROOT)

# ============================================================
# 1. 각 실험 학습
# ============================================================

for exp_name, aug in tqdm(
    EXPERIMENTS.items(),
    total=len(EXPERIMENTS),
    desc="전체 증강 실험",
    unit="experiment"
):

    print("\n" + "=" * 65)
    print(f"실험: {exp_name}")
    print("증강:", aug)
    print("=" * 65)

    run_dir = EXP_ROOT / exp_name
    best_path = run_dir / "weights" / "best.pt"

    if not best_path.exists():
        # 매 실험마다 동일한 기존 가중치에서 시작
        model = YOLO(str(START_WEIGHTS))

        model.train(
            data=str(DATA_YAML),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            device=0,
            workers=2,

            optimizer="AdamW",
            lr0=0.0005,
            lrf=0.1,

            seed=SEED,
            deterministic=True,
            patience=EPOCHS,

            # 실험 간 유일하게 달라지는 설정
            hsv_h=aug["hsv_h"],
            hsv_s=aug["hsv_s"],
            hsv_v=aug["hsv_v"],

            amp=True,
            cache=False,
            val=True,
            save=True,
            plots=False,

            project=str(EXP_ROOT),
            name=exp_name,
            exist_ok=True
        )

        del model
        torch.cuda.empty_cache()

    else:
        print("기존 best.pt 발견, 학습 재실행 생략")

    assert best_path.exists(), best_path

    # ========================================================
    # 2. 같은 Validation 데이터로 평가
    # ========================================================

    trained = YOLO(str(best_path))

    metrics = trained.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=IMGSZ,
        batch=BATCH,
        device=0,
        conf=0.001,
        iou=0.7,
        plots=False,
        verbose=False,
        project=str(EXP_ROOT),
        name=f"{exp_name}_validation",
        exist_ok=True
    )

    # Ultralytics가 제공하는 클래스별 F1
    f1_values = np.asarray(
        metrics.box.f1, dtype=float
    )

    ap_indices = np.asarray(
        metrics.box.ap_class_index,
        dtype=int
    )

    f1_by_class = {
        int(cid): float(f1)
        for cid, f1 in zip(ap_indices, f1_values)
    }

    macro_f1 = float(np.mean([
        f1_by_class.get(cid, 0.0)
        for cid in range(5)
    ]))

    row = {
        "experiment": exp_name,
        "epochs": EPOCHS,
        "hsv_h": aug["hsv_h"],
        "hsv_s": aug["hsv_s"],
        "hsv_v": aug["hsv_v"],
        "macro_f1": macro_f1,
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
        "weights_path": str(best_path)
    }

    summary_rows.append(row)

    for cid, name in enumerate(CLASS_NAMES_YOLO):
        class_rows.append({
            "experiment": exp_name,
            "class": name,
            "f1": f1_by_class.get(cid, 0.0)
        })

    # 실험 하나 끝날 때마다 중간 결과 저장
    pd.DataFrame(summary_rows).to_csv(
        EXP_ROOT / "augmentation_summary.csv",
        index=False
    )

    pd.DataFrame(class_rows).to_csv(
        EXP_ROOT / "augmentation_class_f1.csv",
        index=False
    )

    print(
        f"\n{exp_name} 완료 | "
        f"Macro-F1={macro_f1:.4f} | "
        f"mAP50={metrics.box.map50:.4f} | "
        f"mAP50-95={metrics.box.map:.4f}"
    )

    del trained
    torch.cuda.empty_cache()

# ============================================================
# 3. 최종 결과 비교
# ============================================================

print("\n" + "=" * 65)
print("최종 증강 실험 비교")
print("=" * 65)

summary = pd.DataFrame(summary_rows)

control_f1 = summary.loc[
    summary["experiment"] == "control",
    "macro_f1"
].iloc[0]

summary["f1_delta_pp"] = (
    summary["macro_f1"] - control_f1
) * 100

display(
    summary[
        [
            "experiment",
            "hsv_h",
            "hsv_s",
            "hsv_v",
            "macro_f1",
            "f1_delta_pp",
            "map50",
            "map50_95"
        ]
    ]
)

class_df = pd.DataFrame(class_rows)

display(
    class_df.pivot(
        index="class",
        columns="experiment",
        values="f1"
    )
)

summary.to_csv(
    EXP_ROOT / "augmentation_summary.csv",
    index=False
)

print("\n✅ 실제 증강 학습 실험 완료")
print("새 모델과 결과:", EXP_ROOT)


시작 체크포인트: /content/drive/MyDrive/Colab Notebooks/AI개론/261110스터디/multiclass_yolo26n/models/yolo26n_raw_retrain_best.pt
학습 데이터: /content/multiclass_yolo26n/dataset_raw
결과 저장: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/augmentation_ablation


전체 증강 실험:   0%|          | 0/3 [00:00<?, ?experiment/s]


실험: control
증강: {'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0}
Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/augmentation_ablation/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=12, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.1, mask_ratio=4, max_

   experiment  hsv_h  hsv_s  hsv_v  macro_f1  f1_delta_pp     map50  map50_95
0     control  0.000    0.0   0.00  0.703426     0.000000  0.742787  0.644388
1  brightness  0.000    0.0   0.15  0.716340     1.291420  0.751638  0.652636
2    mild_hsv  0.005    0.1   0.15  0.711833     0.840677  0.746392  0.646200

experiment   brightness   control  mild_hsv
class                                      
brown_glass    0.730950  0.710991  0.716411
clear_glass    0.732808  0.749953  0.750209
green_glass    0.651860  0.610569  0.637771
styrofoam      0.866531  0.848653  0.855334
vinyl          0.599553  0.596965  0.599440


✅ 실제 증강 학습 실험 완료
새 모델과 결과: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/augmentation_ablation


# 기존 TAR + AI Hub JSON에서 Multi-task ROI 준비

In [ ]:

import json
import random
import unicodedata
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

# 원본: 읽기 전용
DATA_ROOT = Path(RAW_ROOT)
SOURCE_LABELS = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "AI개론/261110스터디/labels"
)

# 결과: AI응용
DROP_DIR = Path(CAM_DIR).parent / "dropout_ablation"
DROP_DIR.mkdir(parents=True, exist_ok=True)

# ROI 이미지: Colab 로컬 캐시
ROI_CACHE = Path("/content/dropout_roi_cache")
ROI_CACHE.mkdir(parents=True, exist_ok=True)

MANIFEST = DROP_DIR / "dropout_roi_manifest.csv"

HEAD_DIMS = {
    "class_id": 5,
    "damage": 2,
    "dirtiness": 2,
    "cover": 2,
    "transparency": 2
}

CLASS_INFO = {
    0: ("styrofoam", "05", "001"),
    1: ("vinyl", "06", "001"),
    2: ("brown_glass", "07", "001"),
    3: ("green_glass", "07", "002"),
    4: ("clear_glass", "07", "003")
}

assert SOURCE_LABELS.is_dir(), SOURCE_LABELS

# 클래스별 원본 JSON 경로 검색
json_dirs = {}

for cid, (name, category, detail) in CLASS_INFO.items():
    matches = [
        d for d in SOURCE_LABELS.iterdir()
        if d.is_dir()
        and f"_{category}." in unicodedata.normalize("NFC", d.name)
        and f"_{detail}." in unicodedata.normalize("NFC", d.name)
    ]

    assert len(matches) == 1, (name, matches)
    json_dirs[name] = matches[0]

# JSON 파일 색인
json_index = {}

for name, folder in json_dirs.items():
    json_index[name] = {
        p.stem: p
        for p in folder.rglob("*.json")
    }

def xyxy_from_annotation(obj):
    pts = np.asarray(obj["POINTS"], dtype=float)

    if obj["SHAPE_TYPE"] == "BOX":
        x, y, w, h = pts.reshape(-1)[:4]
        return np.array([x, y, x+w, y+h], dtype=float)

    if obj["SHAPE_TYPE"] == "POLYGON":
        pts = pts.reshape(-1, 2)
        return np.array([
            pts[:, 0].min(), pts[:, 1].min(),
            pts[:, 0].max(), pts[:, 1].max()
        ], dtype=float)

    raise ValueError(obj["SHAPE_TYPE"])

def box_iou(a, b):
    xx1 = max(a[0], b[0])
    yy1 = max(a[1], b[1])
    xx2 = min(a[2], b[2])
    yy2 = min(a[3], b[3])

    inter = max(0, xx2-xx1) * max(0, yy2-yy1)
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])

    return inter / max(area_a + area_b - inter, 1e-9)

def binary_attributes(obj):
    # 새 실험에서 사용할 명시적 이진 라벨 규칙
    values = {
        key: str(obj.get(key, "")).strip()
        for key in ("DAMAGE", "DIRTINESS", "COVER", "TRANSPARENCY")
    }

    if not all(values.values()):
        raise ValueError(f"속성 누락: {values}")

    return {
        "damage": int(values["DAMAGE"] != "원형"),
        "dirtiness": int(values["DIRTINESS"] != "없음"),
        "cover": int(values["COVER"] != "없음"),
        "transparency": int(
            values["TRANSPARENCY"] in ("투명", "반투명")
        )
    }

# ------------------------------------------------------------
# 1. 한 번만 Manifest 복원
# ------------------------------------------------------------

if MANIFEST.is_file():
    roi_df = pd.read_csv(MANIFEST)
    print("기존 ROI Manifest 재사용")
else:
    records = []

    for split in ("train", "val"):
        image_dir = DATA_ROOT / "images" / split
        label_dir = DATA_ROOT / "labels" / split

        image_paths = sorted(image_dir.glob("*.jpg"))

        for image_path in tqdm(
            image_paths,
            desc=f"ROI Manifest {split}",
            unit="image"
        ):
            prefix, original_stem = image_path.stem.split("__", 1)

            cid = next(
                k for k, info in CLASS_INFO.items()
                if info[0] == prefix
            )

            json_path = json_index[prefix].get(original_stem)
            if json_path is None:
                raise FileNotFoundError(
                    f"원본 JSON 없음: {image_path.name}"
                )

            with json_path.open(encoding="utf-8-sig") as f:
                annotation = json.load(f)

            info = annotation["IMAGE_INFO"]
            W = int(info["IMAGE_WIDTH"])
            H = int(info["IMAGE_HEIGHT"])

            objects = annotation["ANNOTATION_INFO"]
            candidate_boxes = [
                xyxy_from_annotation(obj)
                for obj in objects
            ]

            label_path = (
                label_dir / image_path.name
            ).with_suffix(".txt")

            used = set()

            for object_idx, line in enumerate(
                label_path.read_text().splitlines()
            ):
                parts = line.strip().split()
                if len(parts) != 5:
                    continue

                obj_class = int(float(parts[0]))
                cx, cy, bw, bh = map(float, parts[1:])

                gt_box = np.array([
                    (cx-bw/2)*W, (cy-bh/2)*H,
                    (cx+bw/2)*W, (cy+bh/2)*H
                ])

                available = [
                    (j, box_iou(gt_box, candidate_boxes[j]))
                    for j in range(len(objects))
                    if j not in used
                ]

                assert available, (image_path, object_idx)

                match_idx, match_iou = max(
                    available, key=lambda x: x[1]
                )

                # 다른 객체의 속성을 잘못 붙이는 것 방지
                if match_iou < 0.60:
                    raise RuntimeError(
                        f"JSON-GT 매칭 불확실: "
                        f"{image_path.name}, "
                        f"object={object_idx}, IoU={match_iou:.3f}"
                    )

                used.add(match_idx)
                attrs = binary_attributes(objects[match_idx])

                records.append({
                    "split": split,
                    "image_path": str(image_path),
                    "object_idx": object_idx,
                    "class_id": obj_class,
                    "cx": cx, "cy": cy,
                    "bw": bw, "bh": bh,
                    "match_iou": match_iou,
                    **attrs
                })

    roi_df = pd.DataFrame(records)
    roi_df.to_csv(MANIFEST, index=False)

# 필수 검증
assert len(roi_df) == 7579, (
    f"예상 ROI 7579개, 실제 {len(roi_df)}개"
)

assert (
    (roi_df["split"] == "train").sum() == 6214
)
assert (
    (roi_df["split"] == "val").sum() == 1365
)

for head, count in HEAD_DIMS.items():
    assert roi_df[head].notna().all()
    assert roi_df[head].between(0, count-1).all()

print("ROI Manifest 준비:", len(roi_df))
display(pd.crosstab(roi_df["class_id"], roi_df["split"]))

# ------------------------------------------------------------
# 2. ROI 이미지 생성 (다음 실행부터 재사용)
# ------------------------------------------------------------

def make_roi(image, row, size=224, margin=0.08):
    H, W = image.shape[:2]

    cx, cy = float(row.cx), float(row.cy)
    bw, bh = float(row.bw), float(row.bh)

    x1 = max(0, int(np.floor((cx - bw*(0.5+margin))*W)))
    y1 = max(0, int(np.floor((cy - bh*(0.5+margin))*H)))
    x2 = min(W, int(np.ceil((cx + bw*(0.5+margin))*W)))
    y2 = min(H, int(np.ceil((cy + bh*(0.5+margin))*H)))

    crop = image[y1:y2, x1:x2]
    if crop.size == 0:
        raise RuntimeError("비어 있는 ROI")

    h, w = crop.shape[:2]
    scale = min(size/w, size/h)

    nw = max(1, min(size, round(w*scale)))
    nh = max(1, min(size, round(h*scale)))

    resized = cv2.resize(
        crop, (nw, nh),
        interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    )

    canvas = np.full((size, size, 3), 114, dtype=np.uint8)

    left = (size-nw)//2
    top = (size-nh)//2

    canvas[top:top+nh, left:left+nw] = resized
    return canvas

roi_df = roi_df.copy()
roi_df["roi_path"] = [
    str(
        ROI_CACHE / row.split /
        f"{Path(row.image_path).stem}__obj{int(row.object_idx)}.jpg"
    )
    for row in roi_df.itertuples(index=False)
]

# 같은 이미지는 한 번만 읽어 처리
grouped = roi_df.groupby(["split", "image_path"], sort=False)

for (_, image_path), part in tqdm(
    grouped,
    total=len(grouped),
    desc="ROI 이미지 준비",
    unit="image"
):
    pending = [
        row for row in part.itertuples(index=False)
        if not Path(row.roi_path).is_file()
    ]

    if not pending:
        continue

    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(image_path)

    for row in pending:
        output_path = Path(row.roi_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        roi_image = make_roi(image, row)

        assert cv2.imwrite(
            str(output_path),
            roi_image,
            [cv2.IMWRITE_JPEG_QUALITY, 95]
        )

print("\n✅ Dropout 데이터 준비 완료")
print("Train:", (roi_df["split"] == "train").sum())
print("Val:", (roi_df["split"] == "val").sum())
print("Manifest:", MANIFEST)
print("ROI 캐시:", ROI_CACHE)


ROI Manifest train:   0%|          | 0/2800 [00:00<?, ?image/s]

ROI Manifest val:   0%|          | 0/600 [00:00<?, ?image/s]

ROI Manifest 준비: 7579


split     train  val
class_id            
0          1266  362
1          1793  321
2          1404  319
3           771  173
4           980  190

ROI 이미지 준비:   0%|          | 0/3400 [00:00<?, ?image/s]


✅ Dropout 데이터 준비 완료
Train: 6214
Val: 1365
Manifest: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/dropout_ablation/dropout_roi_manifest.csv
ROI 캐시: /content/dropout_roi_cache


# Dropout 0.0 / 0.2 / 0.4 실제 추가 학습

In [ ]:

# ============================================================
# Cell 11 (Fixed)
# ResNet18 Dropout Ablation: p=0.0 / 0.2 / 0.4
#
# Fixes:
# 1. DataLoader num_workers=0
# 2. 정상 완료 여부 확인 후 재사용
# 3. 중단된 실험은 백업 후 처음부터 재실행
# 4. Epoch별 기록 저장
# 5. 완료 표시 파일 생성
# ============================================================

import gc
import json
import random
import shutil

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18
from sklearn.metrics import f1_score, accuracy_score

# ============================================================
# 0. 실험 설정
# ============================================================

SEED = 42
EPOCHS = 6
PATIENCE = 3
BATCH_SIZE = 32
DROPOUT_VALUES = [0.0, 0.2, 0.4]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

DROP_DIR = Path(DROP_DIR)
DROP_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_PT = (
    Path(MODEL_DIR)
    / "resnet18_multitask_raw_roi7579_v1_best.pt"
)

assert SOURCE_PT.is_file(), SOURCE_PT
assert "roi_df" in globals(), "셀 10을 먼저 실행하세요."

assert len(roi_df) == 7579, (
    f"ROI 수가 다릅니다: {len(roi_df)}"
)

assert all(
    Path(p).is_file()
    for p in roi_df["roi_path"]
), "ROI 이미지 캐시가 없습니다. 셀 10을 다시 실행하세요."

print("Device:", DEVICE)
print("Source weights:", SOURCE_PT)
print("Output:", DROP_DIR)

# ============================================================
# 1. 시드 및 Transform
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    normalize
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    normalize
])

# ============================================================
# 2. ROI Dataset
# ============================================================

class ROIDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]

        with Image.open(row["roi_path"]) as image:
            x = self.transform(image.convert("RGB"))

        targets = {
            head: torch.tensor(
                int(row[head]),
                dtype=torch.long
            )
            for head in HEAD_DIMS
        }

        return x, targets


train_frame = roi_df[
    roi_df["split"] == "train"
].copy()

val_frame = roi_df[
    roi_df["split"] == "val"
].copy()

train_dataset = ROIDataset(
    train_frame, train_transform
)

val_dataset = ROIDataset(
    val_frame, val_transform
)

def make_loaders():
    generator = torch.Generator()
    generator.manual_seed(SEED)

    # FIX: num_workers=0
    # Colab/Jupyter multiprocessing 경고 방지

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=USE_AMP
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=USE_AMP
    )

    return train_loader, val_loader

# ============================================================
# 3. 모델 구조
# ============================================================

class AblationResNet18(nn.Module):
    def __init__(self, dropout_p):
        super().__init__()

        self.backbone = resnet18(weights=None)

        feature_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.heads = nn.ModuleDict({
            name: nn.Sequential(
                nn.Dropout(p=dropout_p),
                nn.Linear(feature_dim, count)
            )
            for name, count in HEAD_DIMS.items()
        })

    def forward(self, x):
        features = self.backbone(x)

        return {
            name: head(features)
            for name, head in self.heads.items()
        }

# ============================================================
# 4. Loss
# ============================================================

loss_fns = {}

for head, n_classes in HEAD_DIMS.items():
    counts = np.bincount(
        train_frame[head].to_numpy(dtype=int),
        minlength=n_classes
    )

    weights = np.sqrt(
        counts.sum()
        / (n_classes * np.maximum(counts, 1))
    )

    weights = np.clip(weights, 0.5, 3.0)
    weights /= weights.mean()

    loss_fns[head] = nn.CrossEntropyLoss(
        weight=torch.tensor(
            weights,
            dtype=torch.float32,
            device=DEVICE
        )
    )

def compute_loss(outputs, targets):
    return sum(
        loss_fns[head](
            outputs[head],
            targets[head]
        )
        for head in HEAD_DIMS
    ) / len(HEAD_DIMS)

# ============================================================
# 5. Validation
# ============================================================

@torch.inference_mode()
def validate(model, loader):
    model.eval()

    truths = {
        head: [] for head in HEAD_DIMS
    }

    preds = {
        head: [] for head in HEAD_DIMS
    }

    total_loss = 0.0
    total_count = 0

    for images, targets in loader:
        images = images.to(
            DEVICE, non_blocking=USE_AMP
        )

        targets = {
            head: tensor.to(
                DEVICE, non_blocking=USE_AMP
            )
            for head, tensor in targets.items()
        }

        outputs = model(images)
        loss = compute_loss(outputs, targets)

        count = images.size(0)

        total_loss += loss.item() * count
        total_count += count

        for head in HEAD_DIMS:
            truths[head].extend(
                targets[head].cpu().tolist()
            )

            preds[head].extend(
                outputs[head].argmax(1).cpu().tolist()
            )

    metrics = {
        "val_loss": total_loss / total_count
    }

    for head, n_classes in HEAD_DIMS.items():
        metrics[f"{head}_f1"] = f1_score(
            truths[head],
            preds[head],
            labels=list(range(n_classes)),
            average="macro",
            zero_division=0
        )

        metrics[f"{head}_accuracy"] = accuracy_score(
            truths[head], preds[head]
        )

    metrics["mean_macro_f1"] = float(np.mean([
        metrics[f"{head}_f1"]
        for head in HEAD_DIMS
    ]))

    return metrics

# ============================================================
# 6. 정상 완료 여부 판정
# ============================================================

def completed_from_history(history_path):
    """
    이전 버전의 정상 종료 기록도 인식한다.

    완료 조건:
    - 마지막 Epoch == EPOCHS
    - 또는 Early Stopping 조건에 도달

    주의: 이력만으로는 모든 종료 상황을 확인할 수
    없지만, 이번 p=0.0의 정상 Early Stopping은
    판별할 수 있다.
    """
    if not history_path.is_file():
        return False

    history = pd.read_csv(history_path)

    if history.empty:
        return False

    if "mean_macro_f1" not in history:
        return False

    last_epoch = int(history["epoch"].iloc[-1])

    if last_epoch >= EPOCHS:
        return True

    scores = history["mean_macro_f1"].to_numpy()

    best_score = float(scores[0])
    bad_epochs = 0

    for score in scores[1:]:
        if score > best_score + 1e-4:
            best_score = float(score)
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= PATIENCE:
            return True

    return False


def save_completed_marker(
    path, name, best_epoch, metrics
):
    data = {
        "experiment": name,
        "completed": True,
        "best_epoch": int(best_epoch),
        "mean_macro_f1": float(
            metrics["mean_macro_f1"]
        )
    }

    with path.open("w", encoding="utf-8") as f:
        json.dump(
            data, f, indent=2,
            ensure_ascii=False
        )

# ============================================================
# 7. 동일 초기 체크포인트
# ============================================================

source_checkpoint = torch.load(
    SOURCE_PT,
    map_location="cpu",
    weights_only=True
)

INITIAL_STATE = source_checkpoint["model_state"]

summary_rows = []
all_history = []

print("\nDropout 실험 시작")
print("총 실험:", len(DROPOUT_VALUES))

# ============================================================
# 8. Dropout 실험
# ============================================================

for p in tqdm(
    DROPOUT_VALUES,
    desc="전체 Dropout 실험",
    unit="model"
):

    set_seed(SEED)

    name = f"dropout_{str(p).replace('.', 'p')}"

    run_dir = DROP_DIR / name
    run_dir.mkdir(parents=True, exist_ok=True)

    best_path = run_dir / "best.pt"
    history_path = run_dir / "history.csv"
    done_path = run_dir / "completed.json"

    train_loader, val_loader = make_loaders()

    model = AblationResNet18(
        dropout_p=p
    ).to(DEVICE)

    model.load_state_dict(
        INITIAL_STATE,
        strict=True
    )

    # --------------------------------------------------------
    # 8-1. 이미 완료한 실험만 건너뛰기
    # --------------------------------------------------------

    is_completed = (
        best_path.is_file()
        and history_path.is_file()
        and (
            done_path.is_file()
            or completed_from_history(history_path)
        )
    )

    if is_completed:
        print(f"\n✅ {name}: 완료 결과 재사용")

        saved = torch.load(
            best_path,
            map_location="cpu",
            weights_only=True
        )

        model.load_state_dict(saved["model_state"])

        best_epoch = int(saved["epoch"])

        history = pd.read_csv(
            history_path
        ).to_dict("records")

    else:

        # ----------------------------------------------------
        # 8-2. 이전에 중단된 실험은 백업
        # ----------------------------------------------------

        if best_path.exists() or history_path.exists():
            backup_dir = (
                run_dir / "interrupted_before_fix"
            )
            backup_dir.mkdir(
                parents=True,
                exist_ok=True
            )

            for old_file in [best_path, history_path]:
                backup_file = (
                    backup_dir / old_file.name
                )

                if (
                    old_file.is_file()
                    and not backup_file.exists()
                ):
                    shutil.copy2(
                        old_file, backup_file
                    )

            print(
                f"\n⚠️ {name}: 이전 중단 기록 백업 완료"
            )

        print("\n" + "=" * 60)
        print(f"Dropout p={p} 학습 시작")
        print("=" * 60)

        optimizer = optim.AdamW(
            [
                {
                    "params": model.backbone.parameters(),
                    "lr": 1e-5
                },
                {
                    "params": model.heads.parameters(),
                    "lr": 5e-5
                }
            ],
            weight_decay=1e-4
        )

        scheduler = (
            optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=EPOCHS
            )
        )

        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=USE_AMP
        )

        # ----------------------------------------------------
        # 8-3. Epoch 0
        # ----------------------------------------------------

        initial_metrics = validate(
            model, val_loader
        )

        best_score = initial_metrics[
            "mean_macro_f1"
        ]

        best_epoch = 0
        bad_epochs = 0

        history = [{
            "epoch": 0,
            "train_loss": np.nan,
            **initial_metrics
        }]

        torch.save(
            {
                "model_state": model.state_dict(),
                "epoch": 0,
                "dropout_p": p,
                "val_metrics": initial_metrics
            },
            best_path
        )

        pd.DataFrame(history).to_csv(
            history_path,
            index=False
        )

        print(
            f"{name} | Epoch 0 | "
            f"Mean-F1={best_score:.4f}"
        )

        # ----------------------------------------------------
        # 8-4. 실제 추가 학습
        # ----------------------------------------------------

        for epoch in range(1, EPOCHS + 1):
            model.train()

            loss_sum = 0.0
            seen = 0

            progress = tqdm(
                train_loader,
                desc=f"{name} Epoch {epoch}/{EPOCHS}",
                leave=False,
                unit="batch"
            )

            for images, targets in progress:

                images = images.to(
                    DEVICE,
                    non_blocking=USE_AMP
                )

                targets = {
                    head: tensor.to(
                        DEVICE,
                        non_blocking=USE_AMP
                    )
                    for head, tensor in targets.items()
                }

                optimizer.zero_grad(
                    set_to_none=True
                )

                with torch.autocast(
                    device_type=DEVICE.type,
                    dtype=torch.float16,
                    enabled=USE_AMP
                ):
                    outputs = model(images)
                    loss = compute_loss(
                        outputs, targets
                    )

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                count = images.size(0)

                loss_sum += loss.item() * count
                seen += count

                progress.set_postfix(
                    loss=f"{loss_sum/seen:.4f}"
                )

            scheduler.step()

            val_metrics = validate(
                model, val_loader
            )

            score = val_metrics[
                "mean_macro_f1"
            ]

            history.append({
                "epoch": epoch,
                "train_loss": loss_sum / seen,
                **val_metrics
            })

            pd.DataFrame(history).to_csv(
                history_path,
                index=False
            )

            print(
                f"{name} | Epoch {epoch} | "
                f"Mean-F1={score:.4f} | "
                f"Class-F1="
                f"{val_metrics['class_id_f1']:.4f}"
            )

            if score > best_score + 1e-4:
                best_score = score
                best_epoch = epoch
                bad_epochs = 0

                torch.save(
                    {
                        "model_state": model.state_dict(),
                        "epoch": epoch,
                        "dropout_p": p,
                        "val_metrics": val_metrics
                    },
                    best_path
                )
            else:
                bad_epochs += 1

            if bad_epochs >= PATIENCE:
                print(
                    f"{name}: Early Stopping"
                )
                break

        del optimizer, scheduler, scaler

    # --------------------------------------------------------
    # 8-5. 최적 가중치 평가
    # --------------------------------------------------------

    best_ckpt = torch.load(
        best_path,
        map_location="cpu",
        weights_only=True
    )

    model.load_state_dict(
        best_ckpt["model_state"]
    )

    best_epoch = int(best_ckpt["epoch"])

    best_metrics = validate(
        model, val_loader
    )

    # 최종 평가까지 완료한 경우에만 완료 표시
    save_completed_marker(
        done_path,
        name,
        best_epoch,
        best_metrics
    )

    summary_rows.append({
        "dropout": p,
        "best_epoch": best_epoch,
        **best_metrics,
        "best_path": str(best_path)
    })

    for item in history:
        all_history.append({
            "dropout": p,
            **item
        })

    pd.DataFrame(summary_rows).to_csv(
        DROP_DIR / "dropout_comparison.csv",
        index=False
    )

    print(
        f"\n✅ {name} 완료 | "
        f"Best Epoch={best_epoch} | "
        f"Mean-F1="
        f"{best_metrics['mean_macro_f1']:.4f}"
    )

    del model, train_loader, val_loader
    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# ============================================================
# 9. 최종 결과 비교
# ============================================================

summary = pd.DataFrame(summary_rows)
history_df = pd.DataFrame(all_history)

history_df.to_csv(
    DROP_DIR / "dropout_all_history.csv",
    index=False
)

columns = [
    "dropout",
    "best_epoch",
    "class_id_f1",
    "damage_f1",
    "dirtiness_f1",
    "cover_f1",
    "transparency_f1",
    "mean_macro_f1"
]

print("\n========== 최종 Dropout 비교 ==========")
display(summary[columns])

winner = summary.loc[
    summary["mean_macro_f1"].idxmax()
]

print("\n최적 Dropout:", winner["dropout"])
print("최적 Epoch:", int(winner["best_epoch"]))
print(
    "Class F1:",
    f"{winner['class_id_f1']:.4f}"
)
print(
    "5-head Mean Macro-F1:",
    f"{winner['mean_macro_f1']:.4f}"
)
print("최적 모델:", winner["best_path"])

# ============================================================
# 10. 그래프 저장
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    summary["dropout"],
    summary["mean_macro_f1"] * 100,
    marker="o",
    label="5-head Mean Macro-F1"
)

ax.plot(
    summary["dropout"],
    summary["class_id_f1"] * 100,
    marker="s",
    label="Class Macro-F1"
)

ax.set_xlabel("Dropout probability")
ax.set_ylabel("F1 (%)")
ax.set_title("ResNet18 Dropout Ablation")

ax.set_xticks(DROPOUT_VALUES)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()

plot_path = (
    DROP_DIR / "dropout_f1_comparison.png"
)

fig.savefig(plot_path, dpi=180)
plt.show()
plt.close(fig)

print("\n✅ 전체 Dropout 실험 정상 종료!")
print("결과:", DROP_DIR)


Device: cuda
Source weights: /content/drive/MyDrive/Colab Notebooks/AI개론/261110스터디/multiclass_yolo26n/models/resnet18_multitask_raw_roi7579_v1_best.pt
Output: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/dropout_ablation

Dropout 실험 시작
총 실험: 3


전체 Dropout 실험:   0%|          | 0/3 [00:00<?, ?model/s]


✅ dropout_0p0: 완료 결과 재사용

✅ dropout_0p0 완료 | Best Epoch=1 | Mean-F1=0.7532

⚠️ dropout_0p2: 이전 중단 기록 백업 완료

Dropout p=0.2 학습 시작
dropout_0p2 | Epoch 0 | Mean-F1=0.7153


dropout_0p2 Epoch 1/6:   0%|          | 0/195 [00:00<?, ?batch/s]

dropout_0p2 | Epoch 1 | Mean-F1=0.7460 | Class-F1=0.9446


dropout_0p2 Epoch 2/6:   0%|          | 0/195 [00:00<?, ?batch/s]

... (20 outputs left out)

dropout_0p4 | Epoch 6 | Mean-F1=0.7506 | Class-F1=0.9532

✅ dropout_0p4 완료 | Best Epoch=4 | Mean-F1=0.7519

========== 최종 Dropout 비교 ==========


   dropout  best_epoch  class_id_f1  damage_f1  dirtiness_f1  cover_f1  \
0      0.0           1     0.950988   0.695259           0.5  0.728890   
1      0.2           4     0.954817   0.694620           0.5  0.711872   
2      0.4           4     0.956110   0.690371           0.5  0.716698   

   transparency_f1  mean_macro_f1  
0         0.890783       0.753184  
1         0.893100       0.750882  
2         0.896204       0.751877  


최적 Dropout: 0.0
최적 Epoch: 1
Class F1: 0.9510
5-head Mean Macro-F1: 0.7532
최적 모델: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/dropout_ablation/dropout_0p0/best.pt



✅ 전체 Dropout 실험 정상 종료!
결과: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/dropout_ablation


# CAM → Pseudo Mask

In [ ]:

# ============================================================
# Cell 12. Grad-CAM -> Pseudo Segmentation Mask
# 기존 Kornia ResNet18 체크포인트 재사용
# Validation 시각화 전용 (학습에 사용하지 않음)
# ============================================================

import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0. 경로와 모델
# ------------------------------------------------------------

PSEUDO_DIR = Path(CAM_DIR).parent / "pseudo_masks"
MASK_DIR = PSEUDO_DIR / "masks"
OVERLAY_DIR = PSEUDO_DIR / "overlays"

for folder in (MASK_DIR, OVERLAY_DIR):
    folder.mkdir(parents=True, exist_ok=True)

KORNIA_PT = (
    Path(MODEL_DIR)
    / "resnet18_multitask_kornia_gpu_roi7579_v1_best.pt"
)

assert KORNIA_PT.is_file(), KORNIA_PT
assert "paired" in globals(), "셀 8을 먼저 실행하세요."
assert "GradCAM" in globals()
assert "prepare_roi" in globals()

# 이전 셀에서 정의한 모델 클래스 재사용
model = AblationResNet18(dropout_p=0.2)

ckpt = torch.load(
    KORNIA_PT,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(
    ckpt["model_state"],
    strict=True
)

model = model.to(DEVICE).eval()

cam_engine = GradCAM(model)

CAM_THRESHOLD = 0.45
CLASS_IDS = sorted(CLASS_NAMES.keys())

print("모델:", KORNIA_PT.name)
print("CAM Threshold:", CAM_THRESHOLD)
print("저장:", PSEUDO_DIR)

# ------------------------------------------------------------
# 1. 클래스별 어려운 사례 선정
# ------------------------------------------------------------

selected = []

for cid in CLASS_IDS:

    part = paired[
        paired["gt_id"] == cid
    ].copy()

    # 틀린 사례: 높은 확신도로 틀린 순서
    wrong = (
        part[~part["Kornia_correct"]]
        .sort_values(
            "Kornia_confidence",
            ascending=False
        )
        .head(2)
        .copy()
    )
    wrong["case_type"] = "misclassified"

    # 맞힌 사례: 확신이 낮은 순서
    correct = (
        part[part["Kornia_correct"]]
        .sort_values(
            "Kornia_confidence",
            ascending=True
        )
        .head(1)
        .copy()
    )
    correct["case_type"] = "low_confidence"

    selected.extend([wrong, correct])

cases = pd.concat(
    selected, ignore_index=True
)

print("선정된 ROI:", len(cases))

# ------------------------------------------------------------
# 2. 마스크 생성
# ------------------------------------------------------------

def create_pseudo_mask(cam, threshold=0.45):

    mask = (
        cam >= threshold
    ).astype(np.uint8) * 255

    # 작은 잡음 제거
    kernel3 = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel3
    )

    # 작은 구멍 연결
    kernel5 = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel5
    )

    return mask


def letterbox_content_mask(roi, size=224):
    """Letterbox의 회색 패딩이 아닌 실제 이미지 영역."""
    h, w = roi.shape[:2]

    scale = min(size / w, size / h)

    nw = max(1, round(w * scale))
    nh = max(1, round(h * scale))

    left = (size - nw) // 2
    top = (size - nh) // 2

    content = np.zeros(
        (size, size),
        dtype=np.uint8
    )

    content[
        top:top+nh,
        left:left+nw
    ] = 1

    return content


def make_masked_rgb(image_bgr, mask):
    rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    result = np.zeros_like(rgb)
    result[mask > 0] = rgb[mask > 0]

    return result

# ------------------------------------------------------------
# 3. 분석 및 저장
# ------------------------------------------------------------

records = []
preview = {}

try:
    for idx, case in tqdm(
        cases.iterrows(),
        total=len(cases),
        desc="Pseudo Mask 생성",
        unit="ROI"
    ):

        image = cv2.imread(
            str(case["image_path"])
        )

        assert image is not None

        x1, y1, x2, y2 = [
            int(case[c])
            for c in ("x1", "y1", "x2", "y2")
        ]

        roi = image[y1:y2, x1:x2]
        assert roi.size > 0

        display_roi, tensor = prepare_roi(roi)

        gt_id = int(case["gt_id"])
        pred_id = int(case["Kornia_pred_id"])

        # GT 정보를 사용하지 않고 예측 클래스로 CAM 생성
        cam, actual_pred, probs = cam_engine.generate(
            tensor,
            target_class=pred_id
        )

        assert actual_pred == pred_id

        raw_mask = create_pseudo_mask(
            cam, CAM_THRESHOLD
        )

        # 패딩 영역의 활성화 비율 측정
        content = letterbox_content_mask(roi)

        total_positive = int(
            np.count_nonzero(raw_mask)
        )

        padding_positive = int(
            np.count_nonzero(
                (raw_mask > 0) & (content == 0)
            )
        )

        padding_leakage = (
            padding_positive / max(total_positive, 1)
        )

        # 패딩 영역 제거
        mask = raw_mask * content

        content_pixels = int(
            np.count_nonzero(content)
        )

        coverage = (
            np.count_nonzero(mask)
            / max(content_pixels, 1)
        )

        original_rgb = cv2.cvtColor(
            display_roi,
            cv2.COLOR_BGR2RGB
        )

        heatmap_rgb = overlay_cam(
            display_roi, cam
        )

        extracted_rgb = make_masked_rgb(
            display_roi, mask
        )

        class_name = CLASS_NAMES[gt_id]

        # 동일 이미지에 객체 여러 개가 있을 수 있으므로
        # ROI 순번까지 포함한 고유 파일명
        filename = (
            f"{idx:03d}_"
            f"{class_name}_"
            f"obj{int(case['object_idx'])}"
        )

        cv2.imwrite(
            str(MASK_DIR / f"{filename}.png"),
            mask
        )

        cv2.imwrite(
            str(OVERLAY_DIR / f"{filename}.jpg"),
            cv2.cvtColor(
                heatmap_rgb,
                cv2.COLOR_RGB2BGR
            )
        )

        records.append({
            "image": case["image"],
            "object_idx": int(case["object_idx"]),
            "gt_class": class_name,
            "pred_class": CLASS_NAMES[pred_id],
            "correct": bool(case["Kornia_correct"]),
            "case_type": case["case_type"],
            "confidence": float(probs[pred_id]),
            "mask_coverage": float(coverage),
            "padding_leakage": float(padding_leakage),
            "mask_path": str(
                MASK_DIR / f"{filename}.png"
            )
        })

        # 클래스마다 첫 번째 사례를 미리보기로 선택
        if gt_id not in preview:
            preview[gt_id] = (
                original_rgb,
                heatmap_rgb,
                mask,
                extracted_rgb,
                CLASS_NAMES[pred_id],
                bool(case["Kornia_correct"])
            )

finally:
    cam_engine.close()

# ------------------------------------------------------------
# 4. 결과 시각화
# ------------------------------------------------------------

fig, axes = plt.subplots(
    len(preview), 4,
    figsize=(14, 3.4 * len(preview)),
    squeeze=False,
    constrained_layout=True
)

for row_idx, cid in enumerate(sorted(preview)):

    (
        original,
        heatmap,
        mask,
        extracted,
        pred_name,
        correct
    ) = preview[cid]

    images = [
        original,
        heatmap,
        mask,
        extracted
    ]

    headings = [
        "Original ROI",
        "Grad-CAM",
        "Pseudo Mask",
        "Masked ROI"
    ]

    for col, (img, heading) in enumerate(
        zip(images, headings)
    ):
        ax = axes[row_idx, col]

        if col == 2:
            ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        else:
            ax.imshow(img)

        ax.set_title(
            (
                f"{heading}\n"
                f"GT: {CLASS_NAMES[cid]}"
                if col == 0
                else heading
            ),
            fontsize=10
        )

        ax.axis("off")

    axes[row_idx, 3].set_title(
        f"Pred: {pred_name}\n"
        f"{'Correct' if correct else 'Misclassified'}",
        fontsize=10
    )

preview_path = (
    PSEUDO_DIR / "pseudo_mask_5classes_preview.png"
)

fig.savefig(
    preview_path,
    dpi=180,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

# ------------------------------------------------------------
# 5. 요약 저장
# ------------------------------------------------------------

result_df = pd.DataFrame(records)

result_df.to_csv(
    PSEUDO_DIR / "pseudo_mask_analysis.csv",
    index=False
)

summary_df = (
    result_df.groupby("gt_class", sort=False)
    .agg(
        roi_count=("mask_coverage", "size"),
        mean_coverage=("mask_coverage", "mean"),
        mean_padding_leakage=(
            "padding_leakage", "mean"
        )
    )
    .reset_index()
)

display(summary_df)

print("\n✅ CAM Pseudo Mask 실험 완료")
print("미리보기:", preview_path)
print("분석 CSV:", PSEUDO_DIR / "pseudo_mask_analysis.csv")
print("마스크:", MASK_DIR)


모델: resnet18_multitask_kornia_gpu_roi7579_v1_best.pt
CAM Threshold: 0.45
저장: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/pseudo_masks
선정된 ROI: 15


Pseudo Mask 생성:   0%|          | 0/15 [00:00<?, ?ROI/s]

      gt_class  roi_count  mean_coverage  mean_padding_leakage
0    styrofoam          3       0.463043              0.023884
1        vinyl          3       0.532486              0.055391
2  brown_glass          3       0.497073              0.126327
3  green_glass          3       0.414124              0.049196
4  clear_glass          3       0.447779              0.030438


✅ CAM Pseudo Mask 실험 완료
미리보기: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/pseudo_masks/pseudo_mask_5classes_preview.png
분석 CSV: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/pseudo_masks/pseudo_mask_analysis.csv
마스크: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/pseudo_masks/masks


In [ ]:

# ============================================================
# Cell 13. 기존 GT의 Polygon / BOX 분포 조사
# AI응용 폴더에 결과 저장
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

required = [
    "roi_df", "json_index",
    "xyxy_from_annotation", "box_iou",
    "CAM_DIR"
]

for name in required:
    assert name in globals(), f"변수 없음: {name}"

OUT = Path(CAM_DIR).parent / "segmentation_feasibility"
OUT.mkdir(parents=True, exist_ok=True)

results = []
groups = roi_df.groupby("image_path", sort=False)

print("[1/2] 기존 ROI와 원본 JSON 어노테이션 대조")

for image_path, part in tqdm(
    groups,
    total=len(groups),
    desc="원본 JSON 검사",
    unit="image"
):
    stem = Path(image_path).stem
    class_prefix, original_stem = stem.split("__", 1)

    json_path = json_index[class_prefix].get(original_stem)

    if json_path is None:
        raise FileNotFoundError(stem)

    with open(json_path, encoding="utf-8-sig") as f:
        data = json.load(f)

    info = data["IMAGE_INFO"]
    W = int(info["IMAGE_WIDTH"])
    H = int(info["IMAGE_HEIGHT"])

    objects = data["ANNOTATION_INFO"]
    boxes = [xyxy_from_annotation(o) for o in objects]

    used = set()

    for row in part.itertuples(index=False):
        cx, cy = float(row.cx), float(row.cy)
        bw, bh = float(row.bw), float(row.bh)

        gt_box = np.array([
            (cx - bw/2) * W,
            (cy - bh/2) * H,
            (cx + bw/2) * W,
            (cy + bh/2) * H
        ])

        available = [
            (i, box_iou(gt_box, box))
            for i, box in enumerate(boxes)
            if i not in used
        ]

        if not available:
            raise RuntimeError(f"매칭할 객체 없음: {stem}")

        match_idx, iou = max(
            available,
            key=lambda item: item[1]
        )

        used.add(match_idx)

        obj = objects[match_idx]
        shape = str(
            obj.get("SHAPE_TYPE", "UNKNOWN")
        ).upper()

        vertices = 0

        if shape == "POLYGON":
            vertices = len(
                np.asarray(obj["POINTS"]).reshape(-1, 2)
            )

        results.append({
            "split": row.split,
            "class_id": int(row.class_id),
            "class_name": CLASS_NAMES[int(row.class_id)],
            "image_path": image_path,
            "object_idx": int(row.object_idx),
            "shape_type": shape,
            "vertices": vertices,
            "match_iou": float(iou),
            "reliable_match": iou >= 0.60
        })

result = pd.DataFrame(results)

assert len(result) == len(roi_df)

print("\n[2/2] 클래스별 라벨 유형 집계")

# 매칭이 불확실한 객체는 별도로 구분
result["annotation_type"] = np.where(
    result["reliable_match"],
    result["shape_type"],
    "UNRELIABLE_MATCH"
)

summary = pd.crosstab(
    [
        result["split"],
        result["class_name"]
    ],
    result["annotation_type"]
)

summary["total"] = summary.sum(axis=1)

if "POLYGON" not in summary.columns:
    summary["POLYGON"] = 0

summary["polygon_ratio"] = (
    summary["POLYGON"] / summary["total"]
)

display(summary)

print("\n전체 어노테이션 유형")
display(result["annotation_type"].value_counts())

print("\nTrain Polygon 수:", int(
    (
        (result["split"] == "train")
        & (result["annotation_type"] == "POLYGON")
    ).sum()
))

print("Validation Polygon 수:", int(
    (
        (result["split"] == "val")
        & (result["annotation_type"] == "POLYGON")
    ).sum()
))

result.to_csv(
    OUT / "annotation_shape_details.csv",
    index=False
)

summary.to_csv(
    OUT / "annotation_shape_summary.csv"
)

print("\n✅ Polygon 조사 완료!")
print("저장 위치:", OUT)


[1/2] 기존 ROI와 원본 JSON 어노테이션 대조


원본 JSON 검사:   0%|          | 0/3400 [00:00<?, ?image/s]


[2/2] 클래스별 라벨 유형 집계


annotation_type     BOX  POLYGON  total  polygon_ratio
split class_name                                      
train brown_glass  1196      208   1404       0.148148
      clear_glass   659      321    980       0.327551
      green_glass   478      293    771       0.380026
      styrofoam     720      546   1266       0.431280
      vinyl         891      902   1793       0.503067
val   brown_glass   261       58    319       0.181818
      clear_glass   136       54    190       0.284211
      green_glass   125       48    173       0.277457
      styrofoam     188      174    362       0.480663
      vinyl         199      122    321       0.380062


전체 어노테이션 유형


annotation_type
BOX        4853
POLYGON    2726
Name: count, dtype: int64


Train Polygon 수: 2270
Validation Polygon 수: 456

✅ Polygon 조사 완료!
저장 위치: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/segmentation_feasibility


# Polygon을 Segmentation GT로 변환

In [ ]:

import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm

# =====================================================
# 설정
# =====================================================

SEG_ROOT = Path(CAM_DIR).parent / "deeplabv3plus_polygon"
SEG_ROOT.mkdir(parents=True, exist_ok=True)

LOCAL_ROOT = Path("/content/deeplabv3plus_polygon_roi")
IMG_SIZE = 320
MARGIN = 0.20

SHAPE_CSV = (
    Path(CAM_DIR).parent
    / "segmentation_feasibility"
    / "annotation_shape_details.csv"
)

assert SHAPE_CSV.is_file(), SHAPE_CSV

shapes = pd.read_csv(SHAPE_CSV)

polygon_rows = shapes[
    (shapes["shape_type"] == "POLYGON")
    & (shapes["reliable_match"] == True)
]

polygon_keys = set(
    zip(
        polygon_rows["image_path"],
        polygon_rows["object_idx"].astype(int)
    )
)

assert len(polygon_keys) == 2726, len(polygon_keys)

records = []

# =====================================================
# ROI Letterbox + Mask Letterbox
# =====================================================

def letterbox_pair(image, mask, size=320):
    h, w = image.shape[:2]

    scale = min(size / w, size / h)

    nw = max(1, min(size, round(w * scale)))
    nh = max(1, min(size, round(h * scale)))

    image = cv2.resize(
        image, (nw, nh),
        interpolation=cv2.INTER_AREA if scale < 1
        else cv2.INTER_LINEAR
    )
    mask = cv2.resize(
        mask, (nw, nh),
        interpolation=cv2.INTER_NEAREST
    )

    out_image = np.full(
        (size, size, 3), 114, dtype=np.uint8
    )
    out_mask = np.full(
        (size, size), 255, dtype=np.uint8
    )

    left = (size - nw) // 2
    top = (size - nh) // 2

    out_image[top:top+nh, left:left+nw] = image
    out_mask[top:top+nh, left:left+nw] = mask

    return out_image, out_mask

# =====================================================
# 기존 ROI를 JSON 객체와 다시 연결
# =====================================================

groups = roi_df.groupby("image_path", sort=False)

for image_path, group in tqdm(
    groups,
    total=len(groups),
    desc="Polygon GT 마스크 생성",
    unit="image"
):
    group = group.sort_values("object_idx")

    selected = group[
        group["object_idx"].apply(
            lambda idx: (image_path, int(idx))
            in polygon_keys
        )
    ]

    if selected.empty:
        continue

    image_path = Path(image_path)
    prefix, original_stem = image_path.stem.split("__", 1)

    json_path = json_index[prefix][original_stem]

    with open(json_path, encoding="utf-8-sig") as f:
        data = json.load(f)

    objects = data["ANNOTATION_INFO"]
    info = data["IMAGE_INFO"]

    jw = int(info["IMAGE_WIDTH"])
    jh = int(info["IMAGE_HEIGHT"])

    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(image_path)

    H, W = image.shape[:2]
    sx, sy = W / jw, H / jh

    boxes = [
        xyxy_from_annotation(obj)
        for obj in objects
    ]

    used = set()
    matches = {}

    # 모든 ROI를 순서대로 매칭해 일대일 대응 유지
    for row in group.itertuples(index=False):
        gt_box = np.array([
            (row.cx - row.bw/2) * jw,
            (row.cy - row.bh/2) * jh,
            (row.cx + row.bw/2) * jw,
            (row.cy + row.bh/2) * jh
        ])

        candidates = [
            (i, box_iou(gt_box, box))
            for i, box in enumerate(boxes)
            if i not in used
        ]

        if not candidates:
            raise RuntimeError(f"매칭 실패: {image_path}")

        idx, iou = max(candidates, key=lambda v: v[1])
        used.add(idx)
        matches[int(row.object_idx)] = (idx, iou)

    for row in selected.itertuples(index=False):
        idx, iou = matches[int(row.object_idx)]

        if iou < 0.60:
            raise RuntimeError(
                f"Polygon 매칭 불확실: {image_path}, {iou}"
            )

        obj = objects[idx]
        if obj["SHAPE_TYPE"].upper() != "POLYGON":
            raise RuntimeError(
                f"Polygon 불일치: {image_path}"
            )

        pts = np.asarray(
            obj["POINTS"], dtype=np.float32
        ).reshape(-1, 2)

        pts[:, 0] *= sx
        pts[:, 1] *= sy

        xmin, ymin = pts.min(axis=0)
        xmax, ymax = pts.max(axis=0)

        mx = max(4, (xmax-xmin) * MARGIN)
        my = max(4, (ymax-ymin) * MARGIN)

        x1 = max(0, int(np.floor(xmin-mx)))
        y1 = max(0, int(np.floor(ymin-my)))
        x2 = min(W, int(np.ceil(xmax+mx)))
        y2 = min(H, int(np.ceil(ymax+my)))

        if x2 <= x1 or y2 <= y1:
            raise RuntimeError("잘못된 Crop")

        crop = image[y1:y2, x1:x2]
        mask = np.zeros(crop.shape[:2], dtype=np.uint8)

        # 다른 라벨 객체는 배경으로 학습하지 않음
        for j, other_box in enumerate(boxes):
            if j == idx:
                continue

            ax1, ay1, ax2, ay2 = other_box
            ax1, ax2 = ax1*sx-x1, ax2*sx-x1
            ay1, ay2 = ay1*sy-y1, ay2*sy-y1

            cv2.rectangle(
                mask,
                (int(np.floor(ax1)), int(np.floor(ay1))),
                (int(np.ceil(ax2)), int(np.ceil(ay2))),
                255,
                thickness=-1
            )

        # 현재 객체의 정확한 Polygon을 정답으로
        local_pts = pts.copy()
        local_pts[:, 0] -= x1
        local_pts[:, 1] -= y1

        class_value = int(row.class_id) + 1

        cv2.fillPoly(
            mask,
            [np.rint(local_pts).astype(np.int32)],
            class_value
        )

        crop, mask = letterbox_pair(
            crop, mask, IMG_SIZE
        )

        if not np.any(mask == class_value):
            raise RuntimeError("Polygon 픽셀이 없습니다.")

        stem = (
            f"{image_path.stem}"
            f"__obj{int(row.object_idx)}"
        )

        image_out = (
            LOCAL_ROOT / row.split / "images" / f"{stem}.jpg"
        )
        mask_out = (
            LOCAL_ROOT / row.split / "masks" / f"{stem}.png"
        )

        image_out.parent.mkdir(parents=True, exist_ok=True)
        mask_out.parent.mkdir(parents=True, exist_ok=True)

        cv2.imwrite(
            str(image_out),
            crop,
            [cv2.IMWRITE_JPEG_QUALITY, 95]
        )
        cv2.imwrite(str(mask_out), mask)

        records.append({
            "split": row.split,
            "class_id": int(row.class_id),
            "class_name": CLASS_NAMES[int(row.class_id)],
            "image_path": str(image_out),
            "mask_path": str(mask_out)
        })

seg_df = pd.DataFrame(records)

assert len(seg_df) == 2726
assert (seg_df["split"] == "train").sum() == 2270
assert (seg_df["split"] == "val").sum() == 456

manifest_path = SEG_ROOT / "polygon_roi_manifest.csv"
seg_df.to_csv(manifest_path, index=False)

print("\n✅ Polygon Segmentation 데이터 준비 완료")
display(pd.crosstab(seg_df["class_name"], seg_df["split"]))
print("Manifest:", manifest_path)

# =====================================================
# 클래스별 샘플 미리보기
# =====================================================

fig, axes = plt.subplots(5, 3, figsize=(11, 17))

for cid in range(5):
    row = seg_df[
        seg_df["class_id"] == cid
    ].iloc[0]

    img = cv2.imread(row.image_path)
    mask = cv2.imread(
        row.mask_path, cv2.IMREAD_GRAYSCALE
    )

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    foreground = mask == cid + 1

    extracted = np.zeros_like(rgb)
    extracted[foreground] = rgb[foreground]

    axes[cid, 0].imshow(rgb)
    axes[cid, 1].imshow(foreground, cmap="gray")
    axes[cid, 2].imshow(extracted)

    for j in range(3):
        axes[cid, j].axis("off")

    axes[cid, 0].set_title(CLASS_NAMES[cid])
    axes[cid, 1].set_title("Polygon Ground Truth")
    axes[cid, 2].set_title("GT Masked ROI")

plt.tight_layout()

preview_path = SEG_ROOT / "polygon_ground_truth_preview.png"
fig.savefig(preview_path, dpi=150)
plt.show()
plt.close(fig)

print("GT 미리보기:", preview_path)


Polygon GT 마스크 생성:   0%|          | 0/3400 [00:00<?, ?image/s]


✅ Polygon Segmentation 데이터 준비 완료


split        train  val
class_name             
brown_glass    208   58
clear_glass    321   54
green_glass    293   48
styrofoam      546  174
vinyl          902  122

Manifest: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/deeplabv3plus_polygon/polygon_roi_manifest.csv


GT 미리보기: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/deeplabv3plus_polygon/polygon_ground_truth_preview.png


# DeepLabV3+ 실제 학습 및 mIoU 평가

In [ ]:

%pip -q install segmentation-models-pytorch

import json
import random
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import (
    Dataset, DataLoader, WeightedRandomSampler
)

# =====================================================
# 설정
# =====================================================

SEED = 42
EPOCHS = 8
PATIENCE = 3
BATCH_SIZE = 4
NUM_CLASSES = 6  # background + 5개 클래스
DEVICE = torch.device("cuda")

RUN_DIR = SEG_ROOT / "deeplabv3plus_resnet50"
RUN_DIR.mkdir(parents=True, exist_ok=True)

BEST_PT = RUN_DIR / "best.pt"
LAST_PT = RUN_DIR / "last.pt"
DONE_JSON = RUN_DIR / "completed.json"

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

assert torch.cuda.is_available()

# =====================================================
# Dataset
# =====================================================

class PolygonDataset(Dataset):
    def __init__(self, frame, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]

        img = cv2.imread(row.image_path)
        mask = cv2.imread(
            row.mask_path, cv2.IMREAD_GRAYSCALE
        )

        if img is None or mask is None:
            raise FileNotFoundError(row.image_path)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.augment and random.random() < 0.5:
            img = cv2.flip(img, 1)
            mask = cv2.flip(mask, 1)

        x = img.astype(np.float32) / 255.0

        x = (
            x - np.array([0.485, 0.456, 0.406],
                         dtype=np.float32)
        ) / np.array(
            [0.229, 0.224, 0.225],
            dtype=np.float32
        )

        x = torch.from_numpy(
            np.ascontiguousarray(x.transpose(2, 0, 1))
        )

        y = torch.from_numpy(
            np.ascontiguousarray(mask.astype(np.int64))
        )

        return x, y

train_df = seg_df[seg_df["split"] == "train"].copy()
val_df = seg_df[seg_df["split"] == "val"].copy()

train_ds = PolygonDataset(train_df, augment=True)
val_ds = PolygonDataset(val_df, augment=False)

# 클래스별 ROI 개수 불균형 완화
counts = train_df["class_id"].value_counts()
weights = train_df["class_id"].map(
    lambda cid: 1.0 / np.sqrt(counts[cid])
).to_numpy(dtype=np.float64)

sampler = WeightedRandomSampler(
    weights=torch.from_numpy(weights),
    num_samples=len(train_df),
    replacement=True
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0, pin_memory=True
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0, pin_memory=True
)

# =====================================================
# 모델
# =====================================================

model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    encoder_output_stride=16,
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
).to(DEVICE)

# L4에서 효율적으로 시작:
# Backbone의 앞부분 고정, layer4와 Decoder 학습
for param in model.encoder.parameters():
    param.requires_grad = False

for param in model.encoder.layer4.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.encoder.layer4.parameters(),
            "lr": 1e-5
        },
        {
            "params": model.decoder.parameters(),
            "lr": 1e-4
        },
        {
            "params": model.segmentation_head.parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss(ignore_index=255)
scaler = torch.amp.GradScaler("cuda")

def segmentation_loss(logits, labels):
    if logits.shape[-2:] != labels.shape[-2:]:
        logits = torch.nn.functional.interpolate(
            logits,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False
        )
    return criterion(logits, labels)

# =====================================================
# 검증: 실제 픽셀 IoU
# =====================================================

@torch.inference_mode()
def evaluate_segmentation(model):
    model.eval()
    confusion = np.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        dtype=np.int64
    )

    for images, masks in tqdm(
        val_loader,
        desc="Validation mIoU",
        unit="batch",
        leave=False
    ):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(images)

        if logits.shape[-2:] != masks.shape[-2:]:
            logits = torch.nn.functional.interpolate(
                logits, size=masks.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        preds = logits.argmax(1)

        valid = masks != 255
        ids = (
            masks[valid] * NUM_CLASSES + preds[valid]
        )

        counts = torch.bincount(
            ids, minlength=NUM_CLASSES**2
        )

        confusion += counts.cpu().numpy().reshape(
            NUM_CLASSES, NUM_CLASSES
        )

    intersection = np.diag(confusion).astype(float)
    union = (
        confusion.sum(axis=0)
        + confusion.sum(axis=1)
        - intersection
    )

    iou = np.divide(
        intersection, union,
        out=np.full(NUM_CLASSES, np.nan),
        where=union > 0
    )

    return float(np.nanmean(iou[1:])), iou

# =====================================================
# 중단 후 재개 / 완료 결과 재사용
# =====================================================

history = []
best_score = -1.0
bad_epochs = 0
start_epoch = 1

if DONE_JSON.exists() and BEST_PT.exists():
    print("✅ 학습 완료 체크포인트 발견. 재학습 생략")

elif LAST_PT.exists():
    state = torch.load(
        LAST_PT, map_location="cpu",
        weights_only=False
    )

    model.load_state_dict(state["model_state"])
    optimizer.load_state_dict(state["optimizer_state"])
    scaler.load_state_dict(state["scaler_state"])

    history = state["history"]
    best_score = state["best_score"]
    bad_epochs = state["bad_epochs"]
    start_epoch = state["epoch"] + 1

    print("중단된 학습 재개:", start_epoch, "Epoch")

# =====================================================
# 학습
# =====================================================

if not (DONE_JSON.exists() and BEST_PT.exists()):

    for epoch in range(start_epoch, EPOCHS+1):
        model.train()

        # BatchNorm의 작은 배치 통계 변동 방지
        model.encoder.eval()

        total_loss = 0.0
        total_samples = 0

        progress = tqdm(
            train_loader,
            desc=f"DeepLabV3+ Epoch {epoch}/{EPOCHS}",
            unit="batch"
        )

        for images, masks in progress:
            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                "cuda", dtype=torch.float16
            ):
                logits = model(images)
                loss = segmentation_loss(logits, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            n = images.size(0)
            total_loss += float(loss.item()) * n
            total_samples += n

            progress.set_postfix(
                loss=f"{total_loss/total_samples:.4f}"
            )

        mean_iou, class_ious = evaluate_segmentation(model)

        record = {
            "epoch": epoch,
            "train_loss": total_loss/total_samples,
            "foreground_miou": mean_iou
        }

        for cid in range(NUM_CLASSES):
            record[
                f"iou_{cid}"
            ] = class_ious[cid]

        history.append(record)

        pd.DataFrame(history).to_csv(
            RUN_DIR / "history.csv",
            index=False
        )

        print(
            f"Epoch {epoch}: "
            f"Foreground mIoU={mean_iou:.4f}"
        )

        if mean_iou > best_score + 1e-4:
            best_score = mean_iou
            bad_epochs = 0

            torch.save(
                {
                    "model_state": model.state_dict(),
                    "epoch": epoch,
                    "foreground_miou": mean_iou,
                    "class_ious": class_ious.tolist()
                },
                BEST_PT
            )
        else:
            bad_epochs += 1

        torch.save(
            {
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scaler_state": scaler.state_dict(),
                "epoch": epoch,
                "best_score": best_score,
                "bad_epochs": bad_epochs,
                "history": history
            },
            LAST_PT
        )

        if bad_epochs >= PATIENCE:
            print("Early Stopping")
            break

    DONE_JSON.write_text(
        json.dumps({
            "completed": True,
            "best_foreground_miou": best_score
        }),
        encoding="utf-8"
    )

# =====================================================
# 최적 체크포인트 평가
# =====================================================

best = torch.load(
    BEST_PT, map_location="cpu",
    weights_only=True
)
model.load_state_dict(best["model_state"])

mean_iou, iou_values = evaluate_segmentation(model)

names = [
    "background",
    "styrofoam",
    "vinyl",
    "brown_glass",
    "green_glass",
    "clear_glass"
]

metrics_df = pd.DataFrame({
    "class": names,
    "IoU": iou_values
})

display(metrics_df)

metrics_df.to_csv(
    RUN_DIR / "final_class_iou.csv",
    index=False
)

print("\n========== DeepLabV3+ 최종 결과 ==========")
print("Best Epoch:", best["epoch"])
print("Foreground mIoU:", f"{mean_iou:.4f}")
print("Model:", BEST_PT)

# =====================================================
# Validation 예측 미리보기: 클래스별 1개
# =====================================================

model.eval()

fig, axes = plt.subplots(5, 3, figsize=(11, 17))

for cid in tqdm(
    range(5),
    desc="예측 마스크 미리보기",
    unit="class"
):
    row = val_df[
        val_df["class_id"] == cid
    ].iloc[0]

    img = cv2.imread(row.image_path)
    gt = cv2.imread(
        row.mask_path, cv2.IMREAD_GRAYSCALE
    )

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    image_tensor, _ = val_ds[
        val_df.index.get_loc(row.name)
    ]

    with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(
                image_tensor.unsqueeze(0).to(DEVICE)
            )

    pred = logits.argmax(1)[0].cpu().numpy()

    if pred.shape != gt.shape:
        pred = cv2.resize(
            pred.astype(np.uint8),
            (gt.shape[1], gt.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

    axes[cid, 0].imshow(rgb)
    axes[cid, 1].imshow(
        gt == cid+1, cmap="gray",
        vmin=0, vmax=1
    )
    axes[cid, 2].imshow(
        pred == cid+1, cmap="gray",
        vmin=0, vmax=1
    )

    axes[cid, 0].set_title(names[cid+1])
    axes[cid, 1].set_title("Polygon GT")
    axes[cid, 2].set_title("DeepLabV3+ Prediction")

    for j in range(3):
        axes[cid, j].axis("off")

plt.tight_layout()

preview_path = RUN_DIR / "segmentation_preview.png"
fig.savefig(preview_path, dpi=160)

plt.show()
plt.close(fig)

print("예측 이미지:", preview_path)
print("✅ DeepLabV3+ 실험 완료")


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

DeepLabV3+ Epoch 1/8:   0%|          | 0/568 [00:00<?, ?batch/s]

Validation mIoU:   0%|          | 0/114 [00:00<?, ?batch/s]

Epoch 1: Foreground mIoU=0.6805


... (22 outputs left out)

         class       IoU
0   background  0.918075
1    styrofoam  0.960150
2        vinyl  0.829862
3  brown_glass  0.675711
4  green_glass  0.734795
5  clear_glass  0.771167


========== DeepLabV3+ 최종 결과 ==========
Best Epoch: 8
Foreground mIoU: 0.7943
Model: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/deeplabv3plus_polygon/deeplabv3plus_resnet50/best.pt


예측 마스크 미리보기:   0%|          | 0/5 [00:00<?, ?class/s]

예측 이미지: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/deeplabv3plus_polygon/deeplabv3plus_resnet50/segmentation_preview.png
✅ DeepLabV3+ 실험 완료


# DeepLabV3+ Segmentation -> ResNet18 Classification

In [ ]:

# ============================================================
# Cell 16
# DeepLabV3+ Segmentation -> ResNet18 Classification
#
# RAW vs Predicted Mask vs Polygon GT Mask
# 기존 .pt 재사용 / 추가 학습 없음
# ============================================================

import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUT = Path(CAM_DIR).parent / "segmentation_classification"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. 기존 체크포인트 로드
# ------------------------------------------------------------

SEG_PT = (
    Path(SEG_ROOT)
    / "deeplabv3plus_resnet50"
    / "best.pt"
)

CLS_PT = (
    Path(MODEL_DIR)
    / "resnet18_multitask_kornia_gpu_roi7579_v1_best.pt"
)

assert SEG_PT.is_file(), SEG_PT
assert CLS_PT.is_file(), CLS_PT
assert "seg_df" in globals()
assert "prepare_roi" in globals()
assert "AblationResNet18" in globals()

print("[1/4] 기존 모델 로드")

# DeepLabV3+: 이미 학습된 가중치만 로드
seg_model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights=None,
    encoder_output_stride=16,
    in_channels=3,
    classes=6,
    activation=None
).to(DEVICE)

seg_ckpt = torch.load(
    SEG_PT,
    map_location="cpu",
    weights_only=True
)

seg_model.load_state_dict(
    seg_ckpt["model_state"],
    strict=True
)
seg_model.eval()

# ResNet18: 기존 Kornia 체크포인트
cls_model = AblationResNet18(
    dropout_p=0.2
).to(DEVICE)

cls_ckpt = torch.load(
    CLS_PT,
    map_location="cpu",
    weights_only=True
)

cls_model.load_state_dict(
    cls_ckpt["model_state"],
    strict=True
)
cls_model.eval()

# ------------------------------------------------------------
# 2. 입력 변환
# ------------------------------------------------------------

MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
).reshape(1, 1, 3)

STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
).reshape(1, 1, 3)

def prepare_seg_input(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    x = rgb.astype(np.float32) / 255.0
    x = (x - MEAN) / STD

    return torch.from_numpy(
        np.ascontiguousarray(
            x.transpose(2, 0, 1)
        )
    ).unsqueeze(0).to(DEVICE)

def apply_neutral_mask(bgr, binary_mask):
    result = bgr.copy()
    result[~binary_mask] = (114, 114, 114)
    return result

@torch.inference_mode()
def get_seg_prediction(bgr):
    x = prepare_seg_input(bgr)

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=DEVICE.type == "cuda"
    ):
        logits = seg_model(x)

    if logits.shape[-2:] != bgr.shape[:2]:
        logits = torch.nn.functional.interpolate(
            logits,
            size=bgr.shape[:2],
            mode="bilinear",
            align_corners=False
        )

    return (
        logits.argmax(1)[0]
        .cpu()
        .numpy()
        .astype(np.uint8)
    )

@torch.inference_mode()
def classify_variants(images):
    tensors = []

    for img in images:
        _, tensor = prepare_roi(img)
        tensors.append(tensor.squeeze(0))

    x = torch.stack(tensors, dim=0)

    logits = cls_model(x)["class_id"]

    return torch.softmax(
        logits, dim=1
    ).cpu().numpy()

# ------------------------------------------------------------
# 3. 동일 Validation Polygon ROI 456개 비교
# ------------------------------------------------------------

val_seg = (
    seg_df[seg_df["split"] == "val"]
    .reset_index(drop=True)
)

assert len(val_seg) == 456, len(val_seg)

records = []
previews = {}

print("\n[2/4] Validation 456개 ROI 평가")

for row in tqdm(
    val_seg.itertuples(index=False),
    total=len(val_seg),
    desc="Segmentation -> Classification",
    unit="ROI"
):
    img = cv2.imread(str(row.image_path))

    gt = cv2.imread(
        str(row.mask_path),
        cv2.IMREAD_GRAYSCALE
    )

    if img is None or gt is None:
        raise FileNotFoundError(row.image_path)

    gt_id = int(row.class_id)

    # Segmentation 예측
    seg_pred = get_seg_prediction(img)

    # 예측값 1~5를 모두 전경으로 취급
    # 정답 클래스 ID는 마스크 생성에 사용하지 않음
    pred_fg = seg_pred != 0

    # GT Mask는 비교용(Oracle)
    gt_fg = (
        (gt >= 1)
        & (gt <= 5)
    )

    pred_masked = apply_neutral_mask(
        img, pred_fg
    )

    gt_masked = apply_neutral_mask(
        img, gt_fg
    )

    variants = {
        "RAW": img,
        "PredMask": pred_masked,
        "GTMask": gt_masked
    }

    probs = classify_variants(
        list(variants.values())
    )

    valid = gt != 255

    intersection = np.count_nonzero(
        pred_fg & gt_fg & valid
    )

    union = np.count_nonzero(
        (pred_fg | gt_fg) & valid
    )

    binary_iou = (
        intersection / union
        if union > 0 else np.nan
    )

    record = {
        "image_path": str(row.image_path),
        "gt_id": gt_id,
        "gt_class": CLASS_NAMES[gt_id],
        "binary_mask_iou_valid": binary_iou
    }

    for i, name in enumerate(variants):
        pred_id = int(np.argmax(probs[i]))

        record[f"{name}_pred_id"] = pred_id
        record[f"{name}_pred_class"] = CLASS_NAMES[pred_id]
        record[f"{name}_confidence"] = float(
            probs[i, pred_id]
        )
        record[f"{name}_correct"] = (
            pred_id == gt_id
        )

    records.append(record)

    if gt_id not in previews:
        previews[gt_id] = {
            "original": img.copy(),
            "pred_fg": pred_fg.copy(),
            "pred_masked": pred_masked.copy(),
            "gt_masked": gt_masked.copy()
        }

result_df = pd.DataFrame(records)

# ------------------------------------------------------------
# 4. F1 평가
# ------------------------------------------------------------

print("\n[3/4] 모델별 F1 비교")

summary_rows = []
perclass_rows = []

y_true = result_df["gt_id"].to_numpy()

for name in ("RAW", "PredMask", "GTMask"):
    pred = result_df[
        f"{name}_pred_id"
    ].to_numpy()

    macro_f1 = f1_score(
        y_true, pred,
        labels=list(range(5)),
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true, pred
    )

    summary_rows.append({
        "variant": name,
        "num_roi": len(result_df),
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "correct": int((y_true == pred).sum())
    })

    report = classification_report(
        y_true, pred,
        labels=list(range(5)),
        target_names=[
            CLASS_NAMES[i] for i in range(5)
        ],
        output_dict=True,
        zero_division=0
    )

    for cid in range(5):
        cname = CLASS_NAMES[cid]

        perclass_rows.append({
            "variant": name,
            "class": cname,
            "precision": report[cname]["precision"],
            "recall": report[cname]["recall"],
            "f1": report[cname]["f1-score"]
        })

summary = pd.DataFrame(summary_rows)
perclass = pd.DataFrame(perclass_rows)

raw_f1 = summary.loc[
    summary["variant"] == "RAW",
    "macro_f1"
].iloc[0]

summary["delta_vs_raw_pp"] = (
    summary["macro_f1"] - raw_f1
) * 100

display(summary)

display(
    perclass.pivot(
        index="class",
        columns="variant",
        values="f1"
    )
)

print(
    "\n평균 Binary Mask IoU (유효 픽셀):",
    f"{result_df['binary_mask_iou_valid'].mean():.4f}"
)

# ------------------------------------------------------------
# 5. 시각화
# ------------------------------------------------------------

print("\n[4/4] 비교 이미지 및 CSV 저장")

fig, axes = plt.subplots(
    5, 4,
    figsize=(14, 17),
    constrained_layout=True
)

for cid in range(5):
    item = previews[cid]

    images = [
        item["original"],
        item["pred_fg"],
        item["pred_masked"],
        item["gt_masked"]
    ]

    titles = [
        f"{CLASS_NAMES[cid]} - RAW",
        "Predicted Foreground",
        "Predicted Mask Applied",
        "Polygon GT Applied"
    ]

    for j, (image, title) in enumerate(
        zip(images, titles)
    ):
        ax = axes[cid, j]

        if j == 1:
            ax.imshow(
                image,
                cmap="gray",
                vmin=0,
                vmax=1
            )
        else:
            ax.imshow(
                cv2.cvtColor(
                    image,
                    cv2.COLOR_BGR2RGB
                )
            )

        ax.set_title(title, fontsize=10)
        ax.axis("off")

preview_path = (
    OUT / "segmentation_classification_preview.png"
)

fig.savefig(
    preview_path,
    dpi=160,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

result_df.to_csv(
    OUT / "paired_predictions.csv",
    index=False
)

summary.to_csv(
    OUT / "classification_summary.csv",
    index=False
)

perclass.to_csv(
    OUT / "classification_per_class.csv",
    index=False
)

print("\n========== 최종 결과 ==========")

for row in summary.itertuples(index=False):
    print(
        f"{row.variant:8s} | "
        f"Macro-F1={row.macro_f1:.4f} | "
        f"Accuracy={row.accuracy:.4f} | "
        f"RAW 대비={row.delta_vs_raw_pp:+.2f}%p"
    )

print("\n✅ 실험 완료")
print("저장 경로:", OUT)
print("미리보기:", preview_path)


[1/4] 기존 모델 로드

[2/4] Validation 456개 ROI 평가


Segmentation -> Classification:   0%|          | 0/456 [00:00<?, ?ROI/s]


[3/4] 모델별 F1 비교


    variant  num_roi  accuracy  macro_f1  correct  delta_vs_raw_pp
0       RAW      456  0.969298  0.950188      442         0.000000
1  PredMask      456  0.923246  0.895353      421        -5.483516
2    GTMask      456  0.938596  0.922006      428        -2.818209

variant        GTMask  PredMask       RAW
class                                    
brown_glass  0.887097  0.845528  0.898305
clear_glass  0.875000  0.851064  0.952381
green_glass  0.926316  0.857143  0.907216
styrofoam    0.963989  0.961326  0.997118
vinyl        0.957627  0.961702  0.995918


평균 Binary Mask IoU (유효 픽셀): 0.8856

[4/4] 비교 이미지 및 CSV 저장



========== 최종 결과 ==========
RAW      | Macro-F1=0.9502 | Accuracy=0.9693 | RAW 대비=+0.00%p
PredMask | Macro-F1=0.8954 | Accuracy=0.9232 | RAW 대비=-5.48%p
GTMask   | Macro-F1=0.9220 | Accuracy=0.9386 | RAW 대비=-2.82%p

✅ 실험 완료
저장 경로: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/segmentation_classification
미리보기: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/segmentation_classification/segmentation_classification_preview.png


In [ ]:

# ============================================================
# Cell 17. Mask-aware ResNet18 Fine-tuning
# RAW-only vs RAW/GTMask mixed training
# 기존 .pt 재사용, 3 Epoch 고정
# ============================================================

import cv2
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score

SEED = 42
EPOCHS = 3
BATCH = 32

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

FT_DIR = (
    Path(CAM_DIR).parent
    / "segmentation_classification"
    / "mask_aware_finetune"
)
FT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = Path("/content/segmentation_val_mask_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CLS_PT = (
    Path(MODEL_DIR)
    / "resnet18_multitask_kornia_gpu_roi7579_v1_best.pt"
)

assert CLS_PT.is_file(), CLS_PT
assert "seg_df" in globals()
assert "prepare_roi" in globals()
assert "get_seg_prediction" in globals()
assert "AblationResNet18" in globals()

train_rows = (
    seg_df[seg_df["split"] == "train"]
    .reset_index(drop=True)
)
val_rows = (
    seg_df[seg_df["split"] == "val"]
    .reset_index(drop=True)
)

assert len(train_rows) == 2270
assert len(val_rows) == 456

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# 1. Validation 예측 마스크를 한 번만 생성
# ============================================================

print("[1/4] Validation 예측 마스크 캐시")

seg_model.eval()
pred_mask_paths = []

for idx, row in tqdm(
    val_rows.iterrows(),
    total=len(val_rows),
    desc="Validation Mask 캐싱",
    unit="ROI"
):
    path = CACHE_DIR / f"{idx:04d}.png"

    if not path.is_file():
        img = cv2.imread(str(row["image_path"]))
        assert img is not None

        pred = get_seg_prediction(img)
        assert cv2.imwrite(str(path), pred)

    pred_mask_paths.append(str(path))

val_rows = val_rows.copy()
val_rows["pred_mask_path"] = pred_mask_paths

# ============================================================
# 2. Dataset
# ============================================================

def neutral_mask(img, mask):
    out = img.copy()
    foreground = (mask >= 1) & (mask <= 5)
    out[~foreground] = 114
    return out

class MaskAwareDataset(Dataset):
    def __init__(self, frame, variant="RAW", mix_prob=0.0,
                 train=False):
        self.frame = frame.reset_index(drop=True)
        self.variant = variant
        self.mix_prob = mix_prob
        self.train = train

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]

        img = cv2.imread(str(row["image_path"]))
        if img is None:
            raise FileNotFoundError(row["image_path"])

        use_gt = (
            self.train
            and random.random() < self.mix_prob
        )

        if self.variant == "GTMask" or use_gt:
            mask = cv2.imread(
                str(row["mask_path"]),
                cv2.IMREAD_GRAYSCALE
            )
            img = neutral_mask(img, mask)

        elif self.variant == "PredMask":
            mask = cv2.imread(
                str(row["pred_mask_path"]),
                cv2.IMREAD_GRAYSCALE
            )
            # 예측된 모든 비배경 클래스를 전경으로
            img[mask == 0] = 114

        if self.train and random.random() < 0.5:
            img = cv2.flip(img, 1)

        # 기존 셀 16과 동일한 ResNet 전처리
        _, tensor = prepare_roi(img)

        return (
            tensor.squeeze(0).cpu(),
            int(row["class_id"])
        )

def make_loader(frame, variant="RAW", mix_prob=0.0,
                train=False):
    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        MaskAwareDataset(
            frame,
            variant=variant,
            mix_prob=mix_prob,
            train=train
        ),
        batch_size=BATCH,
        shuffle=train,
        generator=generator,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda"
    )

val_loaders = {
    v: make_loader(val_rows, variant=v)
    for v in ("RAW", "PredMask", "GTMask")
}

# ============================================================
# 3. 동일 모델 구조, 동일 초기 가중치
# ============================================================

initial_checkpoint = torch.load(
    CLS_PT,
    map_location="cpu",
    weights_only=True
)

INITIAL_STATE = initial_checkpoint["model_state"]

def build_model():
    model = AblationResNet18(
        dropout_p=0.2
    ).to(DEVICE)

    model.load_state_dict(
        INITIAL_STATE,
        strict=True
    )
    return model

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()

    gt = []
    pred = []

    for images, labels in loader:
        images = images.to(DEVICE)

        logits = model(images)["class_id"]
        predictions = logits.argmax(1).cpu().numpy()

        gt.extend(labels.numpy().tolist())
        pred.extend(predictions.tolist())

    gt = np.asarray(gt)
    pred = np.asarray(pred)

    overall = {
        "accuracy": accuracy_score(gt, pred),
        "macro_f1": f1_score(
            gt, pred,
            labels=list(range(5)),
            average="macro",
            zero_division=0
        )
    }

    per_class = f1_score(
        gt, pred,
        labels=list(range(5)),
        average=None,
        zero_division=0
    )

    return overall, per_class

result_rows = []
class_rows = []

def record_results(name, model):
    for variant, loader in val_loaders.items():
        metrics, per_class = evaluate(model, loader)

        result_rows.append({
            "model": name,
            "validation_input": variant,
            **metrics
        })

        for cid, score in enumerate(per_class):
            class_rows.append({
                "model": name,
                "validation_input": variant,
                "class": CLASS_NAMES[cid],
                "f1": float(score)
            })

def save_results():
    pd.DataFrame(result_rows).to_csv(
        FT_DIR / "classification_comparison.csv",
        index=False
    )
    pd.DataFrame(class_rows).to_csv(
        FT_DIR / "classification_per_class.csv",
        index=False
    )

# 기존 모델 성능: 학습하지 않음
print("\n[2/4] 기존 Kornia 체크포인트 평가")

base_model = build_model()
record_results("Kornia_original", base_model)
save_results()
del base_model

# ============================================================
# 4. 실제 추가 학습
# ============================================================

EXPERIMENTS = {
    "RAW_control": 0.0,
    "Mixed_50_GTMask": 0.5
}

print("\n[3/4] Mask-aware Fine-tuning")

for exp_name, mix_prob in tqdm(
    EXPERIMENTS.items(),
    desc="Fine-tuning",
    unit="model"
):
    seed_all(SEED)

    run_dir = FT_DIR / exp_name
    run_dir.mkdir(parents=True, exist_ok=True)

    pt_path = run_dir / "epoch3.pt"

    model = build_model()

    if not pt_path.is_file():
        # layer4 + 분류 Head만 학습
        for param in model.parameters():
            param.requires_grad = False

        for param in model.backbone.layer4.parameters():
            param.requires_grad = True

        for param in model.heads["class_id"].parameters():
            param.requires_grad = True

        optimizer = torch.optim.AdamW(
            [
                {
                    "params": model.backbone.layer4.parameters(),
                    "lr": 1e-5
                },
                {
                    "params": model.heads["class_id"].parameters(),
                    "lr": 5e-5
                }
            ],
            weight_decay=1e-4
        )

        criterion = nn.CrossEntropyLoss()
        scaler = torch.amp.GradScaler(
            "cuda",
            enabled=DEVICE.type == "cuda"
        )

        train_loader = make_loader(
            train_rows,
            mix_prob=mix_prob,
            train=True
        )

        history = []

        for epoch in range(1, EPOCHS + 1):
            model.train()

            # 고정된 Backbone BN 통계 유지
            model.backbone.eval()
            model.backbone.layer4.train()
            model.heads["class_id"].train()

            loss_sum = 0.0
            seen = 0

            progress = tqdm(
                train_loader,
                desc=f"{exp_name} Epoch {epoch}/{EPOCHS}",
                leave=False,
                unit="batch"
            )

            for images, labels in progress:
                images = images.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad(set_to_none=True)

                with torch.autocast(
                    device_type=DEVICE.type,
                    dtype=torch.float16,
                    enabled=DEVICE.type == "cuda"
                ):
                    logits = model(images)["class_id"]
                    loss = criterion(logits, labels)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                n = len(labels)
                loss_sum += loss.item() * n
                seen += n

                progress.set_postfix(
                    loss=f"{loss_sum/seen:.4f}"
                )

            history.append({
                "epoch": epoch,
                "train_loss": loss_sum/seen
            })

            print(
                f"{exp_name} | Epoch {epoch} | "
                f"Loss={loss_sum/seen:.4f}"
            )

        pd.DataFrame(history).to_csv(
            run_dir / "history.csv",
            index=False
        )

        # 고정 3 Epoch, Validation 모델 선택 없음
        torch.save({
            "model_state": model.state_dict(),
            "epochs": EPOCHS,
            "mix_prob": mix_prob
        }, pt_path)

    else:
        print(f"{exp_name}: 기존 Epoch 3 체크포인트 재사용")

        ckpt = torch.load(
            pt_path, map_location="cpu",
            weights_only=True
        )
        model.load_state_dict(ckpt["model_state"])

    record_results(exp_name, model)
    save_results()

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# ============================================================
# 5. 결과 비교
# ============================================================

print("\n[4/4] 최종 성능 비교")

comparison = pd.DataFrame(result_rows)

display(
    comparison.pivot(
        index="model",
        columns="validation_input",
        values="macro_f1"
    )
)

class_comparison = pd.DataFrame(class_rows)

display(
    class_comparison[
        class_comparison["validation_input"] == "PredMask"
    ].pivot(
        index="class",
        columns="model",
        values="f1"
    )
)

comparison.to_csv(
    FT_DIR / "classification_comparison.csv",
    index=False
)

print("\n✅ Mask-aware Fine-tuning 완료")
print("결과:", FT_DIR)


[1/4] Validation 예측 마스크 캐시


Validation Mask 캐싱:   0%|          | 0/456 [00:00<?, ?ROI/s]


[2/4] 기존 Kornia 체크포인트 평가

[3/4] Mask-aware Fine-tuning


Fine-tuning:   0%|          | 0/2 [00:00<?, ?model/s]

RAW_control Epoch 1/3:   0%|          | 0/71 [00:00<?, ?batch/s]

RAW_control | Epoch 1 | Loss=0.0330


... (9 outputs left out)

Mixed_50_GTMask | Epoch 3 | Loss=0.0653

[4/4] 최종 성능 비교


validation_input    GTMask  PredMask       RAW
model                                         
Kornia_original   0.922006  0.895353  0.950188
Mixed_50_GTMask   0.956230  0.923197  0.947552
RAW_control       0.920887  0.908520  0.942991

model        Kornia_original  Mixed_50_GTMask  RAW_control
class                                                     
brown_glass         0.845528         0.838095     0.836364
clear_glass         0.851064         0.935780     0.942308
green_glass         0.857143         0.865385     0.888889
styrofoam           0.961326         0.997135     0.940541
vinyl               0.961702         0.979592     0.934498


✅ Mask-aware Fine-tuning 완료
결과: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/segmentation_classification/mask_aware_finetune


# 최종 End-to-End Validation 평가

In [ ]:

# ============================================================
# Cell 18. Final End-to-End Validation
# YOLO RAW vs YOLO Brightness vs YOLO + ResNet18
# ============================================================

import cv2
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from ultralytics import YOLO

VAL_ROOT = Path(RAW_ROOT)
SAVE_DIR = (
    Path(CAM_DIR).parent
    / "evaluation"
    / "end_to_end_validation"
)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS = {
    "YOLO_RAW": (
        Path(MODEL_DIR) / "yolo26n_raw_retrain_best.pt"
    ),
    "YOLO_Brightness": (
        Path(CAM_DIR).parent
        / "augmentation_ablation"
        / "brightness"
        / "weights"
        / "best.pt"
    )
}

RESNET_PT = (
    Path(MODEL_DIR)
    / "resnet18_multitask_kornia_gpu_roi7579_v1_best.pt"
)

for path in [*WEIGHTS.values(), RESNET_PT]:
    assert path.exists(), path

assert "prepare_roi" in globals()
assert "AblationResNet18" in globals()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 동일한 추론 설정
CONF = 0.2862353026866913
NMS_IOU = 0.45
MATCH_IOU = 0.50
IMGSZ = 640

NAMES = [
    "styrofoam",
    "vinyl",
    "brown_glass",
    "green_glass",
    "clear_glass"
]

# ------------------------------------------------------------
# 1. ResNet18 기존 체크포인트
# ------------------------------------------------------------

classifier = AblationResNet18(
    dropout_p=0.2
).to(DEVICE)

state = torch.load(
    RESNET_PT,
    map_location="cpu",
    weights_only=True
)
classifier.load_state_dict(state["model_state"])
classifier.eval()

@torch.inference_mode()
def classify_detections(image, boxes):
    tensors = []

    for box in boxes:
        x1, y1, x2, y2 = map(int, box)

        roi = image[y1:y2, x1:x2]

        if roi.size == 0:
            raise ValueError("빈 YOLO 예측 Box")

        _, tensor = prepare_roi(roi)
        tensors.append(tensor.squeeze(0))

    if not tensors:
        return []

    all_outputs = []

    for start in range(0, len(tensors), 64):
        x = torch.stack(
            tensors[start:start+64]
        ).to(DEVICE)

        out = classifier(x)

        batch = []

        for i in range(x.shape[0]):
            item = {}

            for head, logits in out.items():
                prob = torch.softmax(
                    logits[i], dim=0
                )
                item[head] = int(prob.argmax().item())

            batch.append(item)

        all_outputs.extend(batch)

    return all_outputs

# ------------------------------------------------------------
# 2. Validation GT
# ------------------------------------------------------------

image_paths = sorted(
    (VAL_ROOT / "images" / "val").glob("*.jpg")
)

assert len(image_paths) == 600, len(image_paths)

def get_gt(image_path, width, height):
    path = (
        VAL_ROOT / "labels" / "val"
        / f"{image_path.stem}.txt"
    )

    result = []

    for line in path.read_text().splitlines():
        parts = line.strip().split()

        if len(parts) != 5:
            continue

        cid = int(float(parts[0]))
        cx, cy, bw, bh = map(float, parts[1:])

        result.append({
            "class_id": cid,
            "box": [
                (cx - bw/2)*width,
                (cy - bh/2)*height,
                (cx + bw/2)*width,
                (cy + bh/2)*height
            ]
        })

    return result

def compute_iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    intersection = (
        max(0, x2-x1)*max(0, y2-y1)
    )

    area_a = max(0, a[2]-a[0])*max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0])*max(0, b[3]-b[1])

    return intersection / max(
        area_a+area_b-intersection, 1e-9
    )

# ------------------------------------------------------------
# 3. 클래스별 TP, FP, FN 집계
# ------------------------------------------------------------

def evaluate_one_image(gt, predictions):
    matched_gt = set()
    events = []

    # Confidence 높은 예측부터 일대일 매칭
    predictions = sorted(
        predictions,
        key=lambda x: x["confidence"],
        reverse=True
    )

    for pred in predictions:
        best_idx = None
        best_iou = 0.0

        for i, target in enumerate(gt):
            if i in matched_gt:
                continue

            if pred["class_id"] != target["class_id"]:
                continue

            iou = compute_iou(
                pred["box"], target["box"]
            )

            if iou > best_iou:
                best_iou = iou
                best_idx = i

        if best_idx is not None and best_iou >= MATCH_IOU:
            matched_gt.add(best_idx)
            status = "TP"
        else:
            status = "FP"

        events.append({
            **pred,
            "status": status,
            "match_iou": best_iou if status == "TP" else 0.0
        })

    for i, target in enumerate(gt):
        if i not in matched_gt:
            events.append({
                "class_id": target["class_id"],
                "status": "FN",
                "confidence": np.nan,
                "match_iou": 0.0
            })

    return events

# ------------------------------------------------------------
# 4. 모델별 실제 추론
# ------------------------------------------------------------

all_events = []
all_predictions = []

for model_name, weights in WEIGHTS.items():

    print("\n실험:", model_name)
    detector = YOLO(str(weights))

    for image_path in tqdm(
        image_paths,
        desc=model_name,
        unit="image"
    ):
        image = cv2.imread(str(image_path))
        assert image is not None

        H, W = image.shape[:2]
        gt = get_gt(image_path, W, H)

        result = detector.predict(
            source=image,
            conf=CONF,
            iou=NMS_IOU,
            imgsz=IMGSZ,
            device=0 if DEVICE.type == "cuda" else "cpu",
            verbose=False
        )[0]

        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.int().cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()

        predictions = []

        for box, cid, score in zip(
            boxes, classes, scores
        ):
            predictions.append({
                "box": box.tolist(),
                "class_id": int(cid),
                "confidence": float(score)
            })

        # 원래 YOLO 출력은 항상 평가
        base_events = evaluate_one_image(
            gt, predictions
        )

        for event in base_events:
            all_events.append({
                "model": model_name,
                "image": image_path.name,
                **event
            })

        # Brightness 모델의 예측 ROI에 대해서만
        # ResNet18 클래스를 추가로 평가
        if model_name == "YOLO_Brightness":
            attributes = classify_detections(
                image, boxes
            )

            hybrid = []

            for pred, attr in zip(
                predictions, attributes
            ):
                hybrid.append({
                    "box": pred["box"],
                    "class_id": attr["class_id"],
                    "confidence": pred["confidence"],
                    "attributes": attr
                })

                all_predictions.append({
                    "image": image_path.name,
                    "yolo_class": pred["class_id"],
                    "resnet_class": attr["class_id"],
                    "yolo_confidence": pred["confidence"],
                    **{
                        f"resnet_{key}": value
                        for key, value in attr.items()
                    }
                })

            hybrid_events = evaluate_one_image(
                gt, hybrid
            )

            for event in hybrid_events:
                all_events.append({
                    "model": "YOLO_Brightness_ResNet18",
                    "image": image_path.name,
                    **event
                })

    del detector
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

# ------------------------------------------------------------
# 5. Precision / Recall / F1
# ------------------------------------------------------------

events_df = pd.DataFrame(all_events)

rows = []

for model_name in [
    "YOLO_RAW",
    "YOLO_Brightness",
    "YOLO_Brightness_ResNet18"
]:
    for cid, name in enumerate(NAMES):
        part = events_df[
            (events_df["model"] == model_name)
            & (events_df["class_id"] == cid)
        ]

        tp = int((part["status"] == "TP").sum())
        fp = int((part["status"] == "FP").sum())
        fn = int((part["status"] == "FN").sum())

        precision = tp / max(tp+fp, 1)
        recall = tp / max(tp+fn, 1)
        f1 = 2*tp / max(2*tp+fp+fn, 1)

        rows.append({
            "model": model_name,
            "class": name,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })

class_results = pd.DataFrame(rows)

summary = (
    class_results.groupby("model", as_index=False)
    .agg(
        macro_f1=("f1", "mean"),
        TP=("TP", "sum"),
        FP=("FP", "sum"),
        FN=("FN", "sum")
    )
)

summary["micro_precision"] = (
    summary["TP"] /
    (summary["TP"]+summary["FP"]).clip(lower=1)
)

summary["micro_recall"] = (
    summary["TP"] /
    (summary["TP"]+summary["FN"]).clip(lower=1)
)

summary["micro_f1"] = (
    2*summary["TP"] /
    (
        2*summary["TP"]
        +summary["FP"]
        +summary["FN"]
    ).clip(lower=1)
)

print("\n========== End-to-End Validation ==========")
display(summary)

print("\n클래스별 F1")
display(
    class_results.pivot(
        index="class",
        columns="model",
        values="f1"
    )
)

# ------------------------------------------------------------
# 6. 산출물 저장
# ------------------------------------------------------------

summary.to_csv(
    SAVE_DIR / "end_to_end_summary.csv",
    index=False
)

class_results.to_csv(
    SAVE_DIR / "end_to_end_class_metrics.csv",
    index=False
)

events_df.to_csv(
    SAVE_DIR / "end_to_end_match_events.csv",
    index=False
)

pd.DataFrame(all_predictions).to_csv(
    SAVE_DIR / "resnet18_attribute_predictions.csv",
    index=False
)

print("\n✅ End-to-End Validation 완료")
print("저장:", SAVE_DIR)



실험: YOLO_RAW


YOLO_RAW:   0%|          | 0/600 [00:00<?, ?image/s]


실험: YOLO_Brightness


YOLO_Brightness:   0%|          | 0/600 [00:00<?, ?image/s]


========== End-to-End Validation ==========


                      model  macro_f1    TP   FP   FN  micro_precision  \
0           YOLO_Brightness  0.709045  1036  481  329         0.682927   
1  YOLO_Brightness_ResNet18  0.706721  1033  484  332         0.680949   
2                  YOLO_RAW  0.710271  1067  538  298         0.664798   

   micro_recall  micro_f1  
0      0.758974  0.718945  
1      0.756777  0.716863  
2      0.781685  0.718519  


클래스별 F1


model        YOLO_Brightness  YOLO_Brightness_ResNet18  YOLO_RAW
class                                                           
brown_glass         0.708589                  0.716511  0.710791
clear_glass         0.740920                  0.724638  0.759804
green_glass         0.645833                  0.654545  0.616915
styrofoam           0.852041                  0.841310  0.846452
vinyl               0.597843                  0.596600  0.617391


✅ End-to-End Validation 완료
저장: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/evaluation/end_to_end_validation


# 최종 Test 591장 평가

In [ ]:

# ============================================================
# FIX: get_gt()의 Validation 경로 고정 문제 해결
# train / val / test 자동 인식
# ============================================================

from pathlib import Path

def get_gt(image_path, width, height):
    image_path = Path(image_path)

    # images/test -> labels/test
    split = image_path.parent.name

    assert split in ("train", "val", "test"), (
        f"알 수 없는 데이터 분할: {split}"
    )

    label_path = (
        Path(RAW_ROOT)
        / "labels"
        / split
        / f"{image_path.stem}.txt"
    )

    if not label_path.is_file():
        raise FileNotFoundError(
            f"GT 라벨 파일 없음: {label_path}"
        )

    result = []

    for line in label_path.read_text(
        encoding="utf-8"
    ).splitlines():

        parts = line.strip().split()

        if len(parts) != 5:
            continue

        cid = int(float(parts[0]))
        cx, cy, bw, bh = map(float, parts[1:])

        result.append({
            "class_id": cid,
            "box": [
                (cx - bw / 2) * width,
                (cy - bh / 2) * height,
                (cx + bw / 2) * width,
                (cy + bh / 2) * height
            ]
        })

    return result


# Test 라벨 591개 경로 검사
test_images = sorted(
    (Path(RAW_ROOT) / "images" / "test").glob("*.jpg")
)

missing = [
    p.name for p in test_images
    if not (
        Path(RAW_ROOT) / "labels" / "test"
        / f"{p.stem}.txt"
    ).is_file()
]

print("Test 이미지:", len(test_images))
print("GT 라벨 누락:", len(missing))

assert len(test_images) == 591
assert not missing, f"누락 라벨 예시: {missing[:5]}"

print("✅ 경로 수정 완료! 셀 19를 다시 실행하세요.")


Test 이미지: 591
GT 라벨 누락: 0
✅ 경로 수정 완료! 셀 19를 다시 실행하세요.


In [ ]:

# ============================================================
# Cell 19. FINAL TEST - 591 images
# 확정 모델: YOLO26n RAW
# ResNet18은 속성 출력만 기록, 탐지 클래스는 변경하지 않음
# ============================================================

import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm
from ultralytics import YOLO

# 셀 18의 동일 평가 함수 재사용
for name in (
    "RAW_ROOT", "MODEL_DIR",
    "get_gt", "evaluate_one_image",
    "classify_detections"
):
    assert name in globals(), f"셀 18 변수 없음: {name}"

TEST_DIR = (
    Path(CAM_DIR).parent
    / "evaluation"
    / "final_test"
)

PREVIEW_DIR = TEST_DIR / "prediction_previews"
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = (
    Path(MODEL_DIR)
    / "yolo26n_raw_retrain_best.pt"
)

assert MODEL_PATH.is_file(), MODEL_PATH

# Validation에서 확정한 추론 파라미터: 변경 금지
CONF = 0.2862353026866913
NMS_IOU = 0.45
MATCH_IOU = 0.50
IMGSZ = 640

NAMES = [
    "styrofoam",
    "vinyl",
    "brown_glass",
    "green_glass",
    "clear_glass"
]

test_images = sorted(
    (Path(RAW_ROOT) / "images" / "test").glob("*.jpg")
)

assert len(test_images) == 591, len(test_images)

model = YOLO(str(MODEL_PATH))

events = []
prediction_records = []
preview_done = set()

print("최종 모델:", MODEL_PATH)
print("Test 이미지:", len(test_images))
print("출력:", TEST_DIR)

# ============================================================
# 1. 591개 이미지 실제 추론
# ============================================================

for image_path in tqdm(
    test_images,
    desc="Final Test",
    unit="image"
):
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(image_path)

    H, W = image.shape[:2]
    targets = get_gt(image_path, W, H)

    result = model.predict(
        source=image,
        conf=CONF,
        iou=NMS_IOU,
        imgsz=IMGSZ,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False
    )[0]

    boxes = result.boxes.xyxy.cpu().numpy()
    classes = result.boxes.cls.int().cpu().numpy()
    scores = result.boxes.conf.cpu().numpy()

    detections = [
        {
            "box": box.tolist(),
            "class_id": int(cid),
            "confidence": float(conf)
        }
        for box, cid, conf in zip(boxes, classes, scores)
    ]

    # YOLO 예측을 변경하지 않고 평가
    matched = evaluate_one_image(targets, detections)

    for event in matched:
        events.append({
            "image": image_path.name,
            **event
        })

    # ResNet18 속성 예측은 별도 기록
    attributes = classify_detections(image, boxes)

    for index, (det, attr) in enumerate(
        zip(detections, attributes)
    ):
        prediction_records.append({
            "image": image_path.name,
            "prediction_index": index,
            "yolo_class": NAMES[det["class_id"]],
            "yolo_confidence": det["confidence"],
            "x1": det["box"][0],
            "y1": det["box"][1],
            "x2": det["box"][2],
            "y2": det["box"][3],
            **{
                f"resnet_{key}": value
                for key, value in attr.items()
            }
        })

    # 클래스별 예시 이미지 1장씩 저장
    folder_class = image_path.stem.split("__")[0]

    if (
        folder_class in NAMES
        and folder_class not in preview_done
    ):
        canvas = image.copy()

        # GT: 노란색
        for target in targets:
            x1, y1, x2, y2 = map(int, target["box"])

            cv2.rectangle(
                canvas, (x1, y1), (x2, y2),
                (0, 255, 255), 2
            )

        # Prediction: 초록색
        for det in detections:
            x1, y1, x2, y2 = map(int, det["box"])

            cv2.rectangle(
                canvas, (x1, y1), (x2, y2),
                (0, 255, 0), 2
            )

            label = (
                f"{NAMES[det['class_id']]} "
                f"{det['confidence']:.2f}"
            )

            cv2.putText(
                canvas,
                label,
                (x1, max(15, y1-5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (0, 255, 0),
                1
            )

        cv2.imwrite(
            str(PREVIEW_DIR / f"{folder_class}.jpg"),
            canvas
        )

        preview_done.add(folder_class)

# ============================================================
# 2. 클래스별 Precision, Recall, F1
# ============================================================

events_df = pd.DataFrame(events)
prediction_df = pd.DataFrame(prediction_records)

class_rows = []

for cid, name in enumerate(NAMES):
    subset = events_df[
        events_df["class_id"] == cid
    ]

    tp = int((subset["status"] == "TP").sum())
    fp = int((subset["status"] == "FP").sum())
    fn = int((subset["status"] == "FN").sum())

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * tp / max(2*tp + fp + fn, 1)

    class_rows.append({
        "class": name,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

class_df = pd.DataFrame(class_rows)

TP = int(class_df["TP"].sum())
FP = int(class_df["FP"].sum())
FN = int(class_df["FN"].sum())

summary = pd.DataFrame([{
    "model": "YOLO26n_RAW",
    "images": len(test_images),
    "conf": CONF,
    "nms_iou": NMS_IOU,
    "match_iou": MATCH_IOU,
    "TP": TP,
    "FP": FP,
    "FN": FN,
    "macro_f1": class_df["f1"].mean(),
    "micro_precision": TP / max(TP + FP, 1),
    "micro_recall": TP / max(TP + FN, 1),
    "micro_f1": 2*TP / max(2*TP + FP + FN, 1)
}])

# ============================================================
# 3. 산출물 저장
# ============================================================

class_df.to_csv(
    TEST_DIR / "test_class_metrics.csv",
    index=False
)
summary.to_csv(
    TEST_DIR / "test_summary.csv",
    index=False
)
events_df.to_csv(
    TEST_DIR / "test_match_events.csv",
    index=False
)
prediction_df.to_csv(
    TEST_DIR / "test_attribute_predictions.csv",
    index=False
)

# 클래스별 F1 그래프
fig, ax = plt.subplots(figsize=(9, 4))

ax.bar(
    class_df["class"],
    class_df["f1"] * 100
)

ax.set_title("Final Test - YOLO26n RAW")
ax.set_ylabel("F1 (%)")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", labelrotation=25)
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

fig.savefig(
    TEST_DIR / "test_class_f1.png",
    dpi=180
)

plt.show()
plt.close(fig)

print("\n========== FINAL TEST ==========")
display(summary)
display(class_df)

print("\n✅ Test 591장 평가 완료")
print("결과:", TEST_DIR)


최종 모델: /content/drive/MyDrive/Colab Notebooks/AI개론/261110스터디/multiclass_yolo26n/models/yolo26n_raw_retrain_best.pt
Test 이미지: 591
출력: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/evaluation/final_test


Final Test:   0%|          | 0/591 [00:00<?, ?image/s]


========== FINAL TEST ==========


         model  images      conf  nms_iou  match_iou   TP   FP   FN  macro_f1  \
0  YOLO26n_RAW     591  0.286235     0.45        0.5  869  634  378   0.65683   

   micro_precision  micro_recall  micro_f1  
0         0.578177      0.696872     0.632  

         class   TP   FP   FN  precision    recall        f1
0    styrofoam   88   47   68   0.651852  0.564103  0.604811
1        vinyl  240  312  187   0.434783  0.562061  0.490296
2  brown_glass  246  127   63   0.659517  0.796117  0.721408
3  green_glass  122   96   27   0.559633  0.818792  0.664850
4  clear_glass  173   52   33   0.768889  0.839806  0.802784


✅ Test 591장 평가 완료
결과: /content/drive/MyDrive/Colab Notebooks/AI응용/261113_zerowaste_reuse_experiments/experiments_v3/evaluation/final_test
